In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
from shapely.geometry import Point
from pyproj import Transformer

In [ ]:
BASE_DIR = Path.cwd()

PARQUET_FILES = {
    "base_map": BASE_DIR / "alaska_base_map.parquet",
    "extent": BASE_DIR / "north_slope_extent.parquet",
    "assessment_units": BASE_DIR / "north_slope_assessment_units.parquet",
    "seismic_2d": BASE_DIR / "clean_2d_seismic.parquet",
    "seismic_3d_inventory": BASE_DIR / "clean_3d_seismic.parquet",
    "wells": BASE_DIR / "clean_well_locations.parquet",
}

XYZ_DIR = BASE_DIR / "raw_data" / "north_slope_depth_grids"

TARGET_ANALYSIS_CRS = "EPSG:3338"
TARGET_WEB_CRS = "EPSG:4326"

SURFACE_DECIMATE = 8   # increase for lighter files, reduce for more detail
LINE_VERTEX_STEP = 1   # keep every vertex for now

In [ ]:
def load_gdf(fp, fallback_crs=None):
    gdf = gpd.read_parquet(fp).copy()
    if "geometry" not in gdf.columns:
        raise ValueError(f"{fp.name} has no geometry column")
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry")
    if gdf.crs is None and fallback_crs is not None:
        gdf = gdf.set_crs(fallback_crs, allow_override=True)
    return gdf

def ensure_crs(gdf, target_crs):
    if gdf.crs is None:
        raise ValueError("Layer has no CRS")
    if str(gdf.crs) != target_crs:
        return gdf.to_crs(target_crs)
    return gdf.copy()

In [ ]:
layers = {}

for name, fp in PARQUET_FILES.items():
    if fp.exists():
        layers[name] = load_gdf(fp)
        print(f"Loaded {name}: {len(layers[name])} rows | CRS={layers[name].crs}")
    else:
        print(f"Missing: {fp.name}")

In [ ]:
for name in list(layers.keys()):
    layers[name] = ensure_crs(layers[name], TARGET_ANALYSIS_CRS)

for name, gdf in layers.items():
    print(name, gdf.crs)

In [ ]:
import pandas as pd
from shapely.geometry.base import BaseGeometry

def inspect_layer_geometries(gdf, layer_name):
    print(f"\n--- {layer_name} ---")
    print("rows:", len(gdf))
    print("crs:", gdf.crs)

    type_counts = gdf["geometry"].apply(lambda x: type(x).__name__).value_counts(dropna=False)
    print("\npython types in geometry column:")
    print(type_counts)

    null_mask = gdf["geometry"].isna()
    print("\nnull geometry rows:", null_mask.sum())

    non_geom_mask = ~gdf["geometry"].apply(lambda x: isinstance(x, BaseGeometry) or pd.isna(x))
    print("non-geometry rows:", non_geom_mask.sum())

    if non_geom_mask.sum() > 0:
        display(gdf.loc[non_geom_mask].head(10))

    geom_mask = gdf["geometry"].apply(lambda x: isinstance(x, BaseGeometry))
    geom_only = gdf.loc[geom_mask].copy()

    if len(geom_only) > 0:
        print("\ngeometry types:")
        print(geom_only.geometry.geom_type.value_counts(dropna=False))
        print("\nempty geometries:", geom_only.geometry.is_empty.sum())
        print("invalid geometries:", (~geom_only.geometry.is_valid).sum())
    else:
        print("\nNo valid shapely geometries found.")

In [ ]:
for name, gdf in layers.items():
    inspect_layer_geometries(gdf, name)

In [ ]:
from shapely.geometry.base import BaseGeometry

def clean_layer_geometry(gdf, layer_name):
    gdf = gdf.copy()

    # keep only actual shapely geometries
    gdf = gdf[gdf["geometry"].apply(lambda x: isinstance(x, BaseGeometry))].copy()

    # drop empty geometries
    gdf = gdf[~gdf.geometry.is_empty].copy()

    # optionally drop invalid geometries
    gdf = gdf[gdf.geometry.is_valid].copy()

    print(f"{layer_name}: cleaned to {len(gdf)} rows")
    return gdf

for name in list(layers.keys()):
    layers[name] = clean_layer_geometry(layers[name], name)

In [ ]:
def geometry_to_vertices(gdf, layer_name, layer_group, default_depth=0.0):
    rows = []

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        attrs = row.drop(labels=["geometry"]).to_dict()

        if geom.geom_type == "Point":
            coords = [geom.coords[0]]

        elif geom.geom_type == "MultiPoint":
            coords = []
            for part in geom.geoms:
                coords.extend(list(part.coords))

        elif geom.geom_type == "LineString":
            coords = list(geom.coords)[::LINE_VERTEX_STEP]

        elif geom.geom_type == "MultiLineString":
            coords = []
            for part in geom.geoms:
                coords.extend(list(part.coords)[::LINE_VERTEX_STEP])

        elif geom.geom_type == "Polygon":
            coords = list(geom.exterior.coords)[::LINE_VERTEX_STEP]

        elif geom.geom_type == "MultiPolygon":
            coords = []
            for part in geom.geoms:
                coords.extend(list(part.exterior.coords)[::LINE_VERTEX_STEP])

        else:
            continue

        for order_id, (x, y, *rest) in enumerate(coords):
            rows.append({
                "layer_group": layer_group,
                "layer_name": layer_name,
                "feature_id": idx,
                "vertex_order": order_id,
                "feature_type": geom.geom_type,
                "x_3338": x,
                "y_3338": y,
                "depth_m": default_depth,
                **attrs
            })

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    geom_points = gpd.GeoSeries(
        [Point(xy) for xy in zip(out["x_3338"], out["y_3338"])],
        crs=TARGET_ANALYSIS_CRS
    ).to_crs(TARGET_WEB_CRS)

    out["lon"] = geom_points.x.values
    out["lat"] = geom_points.y.values
    return out

In [ ]:
master_2d_parts = []

layer_specs = [
    ("seismic_2d", "seismic", 0.0),
    ("seismic_3d_inventory", "seismic_inventory", 0.0),
    ("wells", "wells", 0.0),
    ("assessment_units", "geology_framework", 0.0),
    ("extent", "extent", 0.0),
]

for layer_name, layer_group, z0 in layer_specs:
    if layer_name in layers:
        part = geometry_to_vertices(
            layers[layer_name],
            layer_name=layer_name,
            layer_group=layer_group,
            default_depth=z0
        )
        part["source_file"] = PARQUET_FILES[layer_name].name
        master_2d_parts.append(part)
        print(layer_name, part.shape)

master_2d = pd.concat(master_2d_parts, ignore_index=True)
master_2d.head()

In [ ]:
def load_xyz_points(file_path):
    rows = []
    with open(file_path, "r") as f:
        for line in f.readlines()[8:]:
            parts = line.strip().split()
            if len(parts) == 5:
                rows.append(parts)

    df = pd.DataFrame(rows, columns=["x_3338", "y_3338", "z_native", "lon", "lat"])
    df = df.apply(pd.to_numeric, errors="coerce").dropna()

    return df

In [ ]:
surface_parts = []

for fp in sorted(XYZ_DIR.glob("*.XYZ")):
    df = load_xyz_points(fp)

    # decimate so files do not get huge
    df = df.iloc[::SURFACE_DECIMATE, :].copy()

    df["surface_name"] = fp.stem
    df["source_file"] = fp.name

    # use z_native as depth/elevation value
    df["depth_m"] = df["z_native"]

    surface_parts.append(df)

    print(fp.name, df.shape)

master_3d = pd.concat(surface_parts, ignore_index=True)
master_3d.head()

In [ ]:
master_2d.to_parquet("north_slope_master_2d_layers.parquet", index=False)
master_3d.to_parquet("north_slope_master_3d_surfaces.parquet", index=False)

print("Saved:")
print(" - north_slope_master_2d_layers.parquet")
print(" - north_slope_master_3d_surfaces.parquet")

In [ ]:
print(master_2d.shape)
print(master_2d.columns.tolist())
display(master_2d.head())

print(master_3d.shape)
print(master_3d.columns.tolist())
display(master_3d.head())

In [ ]:
print(master_2d["layer_name"].value_counts(dropna=False))
print(master_3d["surface_name"].value_counts(dropna=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

# --------------------------------------------------
# settings
# --------------------------------------------------
XYZ_DIR = Path("raw_data/north_slope_depth_grids")   # adjust only if needed
TARGET_ANALYSIS_CRS = "EPSG:3338"
TARGET_WEB_CRS = "EPSG:4326"

# keep only the surfaces you want
XYZ_SURFACES = [
    "NStopo.XYZ",
    "NSLCU.XYZ",
    "NSshublik.XYZ",
    "NSbasement.XYZ",
    "NStopo-LCU.XYZ",
    "NSLCU-Shublik.XYZ",
    "NSshublik-basement.XYZ",
    "NStopo-basement.XYZ",
]

# --------------------------------------------------
# same idea as your depth-grid notebook
# --------------------------------------------------
def load_xyz_grid(file_path):
    rows = []

    with open(file_path, "r") as f:
        lines = f.readlines()

    # your notebook skipped the first 8 lines
    for line in lines[8:]:
        parts = line.strip().split()
        if len(parts) < 3:
            continue

        try:
            x = float(parts[0])
            y = float(parts[1])
            z = float(parts[2])
            rows.append((x, y, z))
        except ValueError:
            continue

    df = pd.DataFrame(rows, columns=["x_3338", "y_3338", "z_native"])

    grid = df.pivot(index="y_3338", columns="x_3338", values="z_native")
    grid = grid.sort_index(ascending=False)

    return grid

def grid_to_long_df(grid, surface_name):
    df_long = (
        grid
        .stack()
        .reset_index()
    )
    df_long.columns = ["y_3338", "x_3338", "z_native"]

    # reorder to match the rest of your project
    df_long = df_long[["x_3338", "y_3338", "z_native"]].copy()
    df_long["surface_name"] = surface_name
    df_long["source_file"] = f"{surface_name}.XYZ"

    return df_long

# --------------------------------------------------
# load all xyz grids
# --------------------------------------------------
grids = {}

for fname in XYZ_SURFACES:
    fp = XYZ_DIR / fname
    if fp.exists():
        surface_name = fp.stem
        grids[surface_name] = load_xyz_grid(fp)
        print(f"Loaded {surface_name}: {grids[surface_name].shape}")
    else:
        print(f"Missing: {fp}")

# --------------------------------------------------
# convert all grids to one long dataframe
# --------------------------------------------------
surface_parts = []

for surface_name, grid in grids.items():
    part = grid_to_long_df(grid, surface_name)
    surface_parts.append(part)

master_3d = pd.concat(surface_parts, ignore_index=True)

# --------------------------------------------------
# add lon / lat from projected x/y
# --------------------------------------------------
geom = gpd.points_from_xy(master_3d["x_3338"], master_3d["y_3338"])
gdf = gpd.GeoDataFrame(master_3d.copy(), geometry=geom, crs=TARGET_ANALYSIS_CRS)
gdf_ll = gdf.to_crs(TARGET_WEB_CRS)

master_3d["lon"] = gdf_ll.geometry.x
master_3d["lat"] = gdf_ll.geometry.y

# --------------------------------------------------
# depth convention
# keep z_native original, but also provide positive-down depth
# --------------------------------------------------
master_3d["depth_m"] = -master_3d["z_native"]

# final column order
master_3d = master_3d[
    ["x_3338", "y_3338", "z_native", "lon", "lat", "surface_name", "source_file", "depth_m"]
].copy()

print(master_3d.shape)
display(master_3d.head())

print(master_3d["surface_name"].value_counts(dropna=False))
print(master_3d.groupby("surface_name")["z_native"].agg(["min", "max", "mean"]))

In [ ]:
master_3d.to_parquet("north_slope_master_3d_surfaces.parquet", index=False)
print("Saved: north_slope_master_3d_surfaces.parquet")


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# -----------------------------
# choose main surfaces only
# -----------------------------
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]

surface_settings = {
    "NStopo": {"color": "#4daf4a", "size": 2, "opacity": 0.35},
    "NSLCU": {"color": "#377eb8", "size": 2, "opacity": 0.35},
    "NSshublik": {"color": "#ff7f00", "size": 2, "opacity": 0.35},
    "NSbasement": {"color": "#984ea3", "size": 2, "opacity": 0.35},
}

fig = go.Figure()

# -----------------------------
# add 3D surfaces as point clouds
# -----------------------------
for name in main_surfaces:
    df = master_3d[master_3d["surface_name"] == name].copy()

    fig.add_trace(go.Scatter3d(
        x=df["x_3338"],
        y=df["y_3338"],
        z=df["depth_m"],
        mode="markers",
        name=name,
        marker=dict(
            size=surface_settings[name]["size"],
            color=surface_settings[name]["color"],
            opacity=surface_settings[name]["opacity"]
        ),
        hovertemplate=(
            f"<b>{name}</b><br>"
            "X: %{x}<br>"
            "Y: %{y}<br>"
            "Depth: %{z}<extra></extra>"
        )
    ))

# -----------------------------
# add wells
# -----------------------------
wells_df = master_2d[master_2d["layer_name"] == "wells"].copy()

fig.add_trace(go.Scatter3d(
    x=wells_df["x_3338"],
    y=wells_df["y_3338"],
    z=[0] * len(wells_df),   # wells shown at surface for now
    mode="markers",
    name="Wells",
    marker=dict(
        size=3,
        color="black",
        opacity=0.9,
        symbol="circle"
    ),
    hovertemplate=(
        "<b>Well</b><br>"
        "X: %{x}<br>"
        "Y: %{y}<br>"
        "Z: %{z}<extra></extra>"
    )
))

fig.update_layout(
    title="North Slope Structural Surfaces + Wells",
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth (m, positive downward)",
        zaxis=dict(autorange="reversed"),
        aspectmode="data",
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=0.8)
        )
    ),
    height=900,
    width=1300,
    legend=dict(itemsizing="constant")
)

fig.show()

In [ ]:
seis3d_df = master_2d[master_2d["layer_name"] == "seismic_3d_inventory"].copy()

fig.add_trace(go.Scatter3d(
    x=seis3d_df["x_3338"],
    y=seis3d_df["y_3338"],
    z=[0] * len(seis3d_df),
    mode="markers",
    name="3D Seismic Inventory",
    marker=dict(
        size=1,
        color="red",
        opacity=0.15
    ),
    hoverinfo="skip"
))

In [ ]:
from pathlib import Path
import subprocess
import sys

# folder where your notebooks are
BASE_DIR = Path.cwd()

# pick the notebooks you want
notebooks = [
    "01_base_map_reorganized.ipynb",
    "01_parquet_prep_cleaned.ipynb",
    "02_geologic_framework.ipynb",
    "Depth_grid_processing.ipynb",
    "North Slope Data Layer Map.ipynb",
    "Master.ipynb",
]

export_dir = BASE_DIR / "notebook_exports"
export_dir.mkdir(exist_ok=True)

for nb in notebooks:
    nb_path = BASE_DIR / nb
    if not nb_path.exists():
        print(f"SKIP: {nb} not found")
        continue

    print(f"Exporting {nb} -> HTML")
    cmd = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "html",
        str(nb_path),
        "--output-dir",
        str(export_dir),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"DONE: {nb}")
    else:
        print(f"FAILED: {nb}")
        print(result.stderr)

print(f"\nFinished. Exported files are in:\n{export_dir}")

In [ ]:
from pathlib import Path
import subprocess
import sys

BASE_DIR = Path.cwd()
export_dir = BASE_DIR / "notebook_exports_pdf"
export_dir.mkdir(exist_ok=True)

notebooks = [
    "01_base_map_reorganized.ipynb",
    "01_parquet_prep_cleaned.ipynb",
    "02_geologic_framework.ipynb",
    "Depth_grid_processing.ipynb",
    "North Slope Data Layer Map.ipynb",
    "Master.ipynb",
]

for nb in notebooks:
    nb_path = BASE_DIR / nb
    if not nb_path.exists():
        print(f"SKIP: {nb} not found")
        continue

    print(f"Exporting {nb} -> PDF")
    cmd = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "pdf",
        str(nb_path),
        "--output-dir",
        str(export_dir),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"DONE: {nb}")
    else:
        print(f"FAILED: {nb}")
        print(result.stderr)

print(f"\nFinished. PDF files are in:\n{export_dir}")

In [ ]:
from pathlib import Path
import shutil

BASE = Path.cwd()

# -----------------------------
# folders to create
# -----------------------------
folders = [
    "01_pipeline",
    "02_visualization",
    "03_data_final/core_layers",
    "03_data_final/feature_layers",
    "03_data_final/master_layers",
    "04_data_intermediate/structural_working",
    "04_data_intermediate/reports",
    "05_exports/html",
    "05_exports/pdf",
    "06_archive/old_notebooks",
]

for folder in folders:
    (BASE / folder).mkdir(parents=True, exist_ok=True)

# -----------------------------
# file routing rules
# -----------------------------
move_map = {
    # notebooks you want to keep active
    "01_rebuild_pipeline_consolidated.ipynb": "01_pipeline",
    "North Slope Data Layer Map.ipynb": "02_visualization",

    # old notebooks -> archive
    "01_base_map_reorganized.ipynb": "06_archive/old_notebooks",
    "01_parquet_prep_cleaned.ipynb": "06_archive/old_notebooks",
    "02_geologic_framework.ipynb": "06_archive/old_notebooks",
    "Depth_grid_processing.ipynb": "06_archive/old_notebooks",
    "Master.ipynb": "06_archive/old_notebooks",

    # final core layers
    "alaska_base_map.parquet": "03_data_final/core_layers",
    "north_slope_extent.parquet": "03_data_final/core_layers",
    "north_slope_assessment_units.parquet": "03_data_final/core_layers",
    "clean_2d_seismic.parquet": "03_data_final/core_layers",
    "clean_3d_seismic.parquet": "03_data_final/core_layers",
    "clean_well_locations.parquet": "03_data_final/core_layers",

    # final feature layers
    "clean_2d_seismic_features.parquet": "03_data_final/feature_layers",
    "clean_3d_seismic_features.parquet": "03_data_final/feature_layers",
    "clean_well_locations_features.parquet": "03_data_final/feature_layers",
    "clean_well_locations_features_dedup.parquet": "03_data_final/feature_layers",
    "north_slope_assessment_units_features.parquet": "03_data_final/feature_layers",
    "north_slope_extent_features.parquet": "03_data_final/feature_layers",

    # final master layers
    "north_slope_master_2d_layers.parquet": "03_data_final/master_layers",
    "north_slope_master_3d_surfaces.parquet": "03_data_final/master_layers",

    # intermediate / working structural files
    "north_slope_structural_surfaces.parquet": "04_data_intermediate/structural_working",
    "north_slope_structural_surfaces_3d_ready.parquet": "04_data_intermediate/structural_working",
    "north_slope_structural_surfaces_3d_ready_dedup.parquet": "04_data_intermediate/structural_working",
    "north_slope_structural_surfaces_in_3d_extent.parquet": "04_data_intermediate/structural_working",

    # reports
    "parquet_inspection_report.txt": "04_data_intermediate/reports",
    "parquet_inspection.txt": "04_data_intermediate/reports",

    # html output
    "north_slope_plotly_advanced.html": "05_exports/html",
}

# -----------------------------
# move exact-match files
# -----------------------------
moved = []
skipped = []

for name, rel_dest in move_map.items():
    src = BASE / name
    dest = BASE / rel_dest / name
    if src.exists():
        if dest.exists():
            skipped.append((name, "destination already exists"))
        else:
            shutil.move(str(src), str(dest))
            moved.append((name, rel_dest))
    else:
        skipped.append((name, "not found"))

# -----------------------------
# move export folders if present
# -----------------------------
export_folders = {
    "notebook_exports": "05_exports/html",
    "notebook_exports_pdf": "05_exports/pdf",
}

for folder_name, rel_dest in export_folders.items():
    src = BASE / folder_name
    dest = BASE / rel_dest / folder_name
    if src.exists():
        if dest.exists():
            skipped.append((folder_name, "destination already exists"))
        else:
            shutil.move(str(src), str(dest))
            moved.append((folder_name, rel_dest))
    else:
        skipped.append((folder_name, "not found"))

# -----------------------------
# summary
# -----------------------------
print("\nMOVED:")
for name, dest in moved:
    print(f"  {name}  ->  {dest}")

print("\nSKIPPED:")
for name, reason in skipped:
    print(f"  {name}: {reason}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go

BASE_DIR = Path.cwd()

CORE_DIR = BASE_DIR / "03_data_final" / "core_layers"
FEATURE_DIR = BASE_DIR / "03_data_final" / "feature_layers"
MASTER_DIR = BASE_DIR / "03_data_final" / "master_layers"
XYZ_DIR = BASE_DIR / "raw_data" / "north_slope_depth_grids"
EXPORT_HTML_DIR = BASE_DIR / "05_exports" / "html"

EXPORT_HTML_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    # final integrated geometry
    "master_2d": MASTER_DIR / "north_slope_master_2d_layers.parquet",
    "master_3d": MASTER_DIR / "north_slope_master_3d_surfaces.parquet",

    # feature enrichment
    "wells_features": FEATURE_DIR / "clean_well_locations_features_dedup.parquet",
    "seis2d_features": FEATURE_DIR / "clean_2d_seismic_features.parquet",
    "seis3d_features": FEATURE_DIR / "clean_3d_seismic_features.parquet",
    "assessment_features": FEATURE_DIR / "north_slope_assessment_units_features.parquet",
    "extent_features": FEATURE_DIR / "north_slope_extent_features.parquet",

    # optional fallback/core
    "wells_core": CORE_DIR / "clean_well_locations.parquet",
    "seis2d_core": CORE_DIR / "clean_2d_seismic.parquet",
    "seis3d_core": CORE_DIR / "clean_3d_seismic.parquet",
    "assessment_core": CORE_DIR / "north_slope_assessment_units.parquet",
    "extent_core": CORE_DIR / "north_slope_extent.parquet",
}

for k, v in PATHS.items():
    print(f"{k}: {v.exists()} -> {v}")
print("XYZ_DIR exists:", XYZ_DIR.exists(), "->", XYZ_DIR)

In [ ]:
from pathlib import Path

BASE = Path.cwd()

print("=" * 100)
print("CURRENT WORKING DIRECTORY")
print("=" * 100)
print(BASE)

print("\n" + "=" * 100)
print("KEY FILES BY TYPE")
print("=" * 100)

patterns = {
    "PARQUETS": "*.parquet",
    "NOTEBOOKS": "*.ipynb",
    "HTML": "*.html",
    "TEXT REPORTS": "*.txt",
    "PDF EXPORT FOLDERS": "*pdf*",
}

for label, pattern in patterns.items():
    print(f"\n--- {label} ---")
    matches = sorted(BASE.rglob(pattern))
    if not matches:
        print("None found")
        continue
    for p in matches:
        try:
            rel = p.relative_to(BASE)
        except Exception:
            rel = p
        print(rel)

print("\n" + "=" * 100)
print("XYZ FILES")
print("=" * 100)
xyz_matches = sorted(BASE.rglob("*.XYZ"))
if not xyz_matches:
    print("No XYZ files found")
else:
    for p in xyz_matches:
        print(p.relative_to(BASE))

print("\n" + "=" * 100)
print("LIKELY IMPORTANT FOLDERS")
print("=" * 100)
folder_names = [
    "raw_data",
    "01_pipeline",
    "02_visualization",
    "03_data_final",
    "04_data_intermediate",
    "05_exports",
    "06_archive",
]

for name in folder_names:
    matches = [p for p in BASE.rglob(name) if p.is_dir()]
    print(f"\n{name}:")
    if matches:
        for p in matches:
            print(" ", p.relative_to(BASE))
    else:
        print("  not found")

print("\n" + "=" * 100)
print("TOP-LEVEL CONTENTS")
print("=" * 100)
for p in sorted(BASE.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"{kind:4}  {p.name}")

print("\n" + "=" * 100)
print("SHALLOW TREE (3 levels)")
print("=" * 100)

def print_tree(root, max_depth=3, prefix=""):
    root = Path(root)
    if max_depth < 0:
        return
    items = sorted(root.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
    for item in items:
        print(prefix + item.name + ("/" if item.is_dir() else ""))
        if item.is_dir() and max_depth > 0:
            print_tree(item, max_depth=max_depth - 1, prefix=prefix + "    ")

print_tree(BASE, max_depth=2)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go

# -----------------------------
# LOAD DATA
# -----------------------------
master_2d = pd.read_parquet(PATHS["master_2d"])
master_3d = pd.read_parquet(PATHS["master_3d"])

wells_feat = gpd.read_parquet(PATHS["wells_features"])
seis2d_feat = gpd.read_parquet(PATHS["seis2d_features"])
seis3d_feat = gpd.read_parquet(PATHS["seis3d_features"])
au_feat = gpd.read_parquet(PATHS["assessment_features"])
extent_feat = gpd.read_parquet(PATHS["extent_features"])

# -----------------------------
# MAIN STRUCTURAL SURFACES ONLY
# -----------------------------
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]
surface_settings = {
    "NStopo": {"opacity": 0.30, "colorscale": "Greens"},
    "NSLCU": {"opacity": 0.40, "colorscale": "Blues"},
    "NSshublik": {"opacity": 0.50, "colorscale": "Cividis"},
    "NSbasement": {"opacity": 0.70, "colorscale": "Viridis"},
}

surf = master_3d[master_3d["surface_name"].isin(main_surfaces)].copy()
z_col = "depth_m" if "depth_m" in surf.columns else "z_native"

fig = go.Figure()

# -----------------------------
# STRUCTURAL SURFACES AS REAL SURFACES
# -----------------------------
for name in main_surfaces:
    df = surf[surf["surface_name"] == name].copy()

    grid = (
        df.pivot_table(index="y_3338", columns="x_3338", values=z_col, aggfunc="mean")
        .sort_index(ascending=False)
        .sort_index(axis=1)
    )

    X = grid.columns.to_numpy()
    Y = grid.index.to_numpy()
    Z = grid.to_numpy()
    X_mesh, Y_mesh = np.meshgrid(X, Y)

    fig.add_trace(
        go.Surface(
            x=X_mesh,
            y=Y_mesh,
            z=Z,
            name=name,
            opacity=surface_settings[name]["opacity"],
            colorscale=surface_settings[name]["colorscale"],
            showscale=False,
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Depth/Elev: %{z:.0f}<extra></extra>"
            ),
        )
    )

# -----------------------------
# WELLS WITH FEATURE HOVER
# -----------------------------
wells_plot = wells_feat.copy()
if "lon" not in wells_plot.columns or "lat" not in wells_plot.columns:
    wells_plot = wells_plot.to_crs("EPSG:4326")
    wells_plot["lon"] = wells_plot.geometry.x
    wells_plot["lat"] = wells_plot.geometry.y

wells_xy = master_2d[master_2d["layer_name"] == "wells"][["x_3338", "y_3338", "lon", "lat"]].copy()
wells_xy = wells_xy.drop_duplicates(subset=["lon", "lat"])

wells_plot = wells_plot.merge(
    wells_xy,
    on=["lon", "lat"],
    how="left",
    suffixes=("", "_m2d")
)

fig.add_trace(
    go.Scatter3d(
        x=wells_plot["x_3338"],
        y=wells_plot["y_3338"],
        z=np.zeros(len(wells_plot)),
        mode="markers",
        name="Wells",
        marker=dict(size=3, color="black", opacity=0.9),
        customdata=np.stack([
            wells_plot.get("operator", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("field", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("au_name", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("dist_to_2d_km", pd.Series([np.nan] * len(wells_plot))).round(2).astype(str),
            wells_plot.get("in_3d", pd.Series([False] * len(wells_plot))).astype(str),
        ], axis=1),
        hovertemplate=(
            "<b>Well</b><br>"
            "Operator: %{customdata[0]}<br>"
            "Field: %{customdata[1]}<br>"
            "AU: %{customdata[2]}<br>"
            "Dist to 2D (km): %{customdata[3]}<br>"
            "Inside 3D coverage: %{customdata[4]}<extra></extra>"
        ),
    )
)

# -----------------------------
# 2D SEISMIC FROM MASTER GEOMETRY
# -----------------------------
seis2d_xy = master_2d[master_2d["layer_name"] == "seismic_2d"].copy()
fig.add_trace(
    go.Scatter3d(
        x=seis2d_xy["x_3338"],
        y=seis2d_xy["y_3338"],
        z=np.zeros(len(seis2d_xy)),
        mode="lines",
        name="2D Seismic",
        line=dict(color="red", width=2),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# -----------------------------
# 3D SEISMIC INVENTORY
# -----------------------------
seis3d_xy = master_2d[master_2d["layer_name"] == "seismic_3d_inventory"].copy()
fig.add_trace(
    go.Scatter3d(
        x=seis3d_xy["x_3338"],
        y=seis3d_xy["y_3338"],
        z=np.zeros(len(seis3d_xy)),
        mode="markers",
        name="3D Seismic Inventory",
        marker=dict(size=1, color="red", opacity=0.08),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# -----------------------------
# ASSESSMENT UNITS
# -----------------------------
au_xy = master_2d[master_2d["layer_name"] == "assessment_units"].copy()
fig.add_trace(
    go.Scatter3d(
        x=au_xy["x_3338"],
        y=au_xy["y_3338"],
        z=np.zeros(len(au_xy)),
        mode="lines",
        name="Assessment Units",
        line=dict(color="orange", width=1),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# -----------------------------
# EXTENT BOX
# -----------------------------
extent_xy = master_2d[master_2d["layer_name"] == "extent"].copy()
fig.add_trace(
    go.Scatter3d(
        x=extent_xy["x_3338"],
        y=extent_xy["y_3338"],
        z=np.zeros(len(extent_xy)),
        mode="lines",
        name="North Slope Extent",
        line=dict(color="black", width=4),
        hoverinfo="skip",
        visible=True,
    )
)

# -----------------------------
# LAYOUT
# -----------------------------
fig.update_layout(
    title="North Slope Master Analysis Scene",
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth / Elevation",
        zaxis=dict(autorange="reversed"),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.25),
        camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)),
    ),
    height=900,
    width=1450,
    legend=dict(itemsizing="constant"),
)
fig.show()

# -----------------------------
# SAVE HTML
# -----------------------------
out_html = EXPORT_HTML_DIR / "north_slope_master_analysis_scene.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"Saved: {out_html.resolve()}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go

BASE_DIR = Path.cwd()

CORE_DIR = BASE_DIR / "03_data_final" / "core_layers"
FEATURE_DIR = BASE_DIR / "03_data_final" / "feature_layers"
MASTER_DIR = BASE_DIR / "03_data_final" / "master_layers"
EXPORT_HTML_DIR = BASE_DIR / "05_exports" / "html"

EXPORT_HTML_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "master_2d": MASTER_DIR / "north_slope_master_2d_layers.parquet",
    "master_3d": MASTER_DIR / "north_slope_master_3d_surfaces.parquet",
    "wells_features": FEATURE_DIR / "clean_well_locations_features_dedup.parquet",
    "seis2d_features": FEATURE_DIR / "clean_2d_seismic_features.parquet",
    "seis3d_features": FEATURE_DIR / "clean_3d_seismic_features.parquet",
    "assessment_features": FEATURE_DIR / "north_slope_assessment_units_features.parquet",
    "extent_features": FEATURE_DIR / "north_slope_extent_features.parquet",
}

for k, v in PATHS.items():
    print(f"{k}: {v.exists()} -> {v}")

# -----------------------------
# LOAD DATA
# -----------------------------
master_2d = pd.read_parquet(PATHS["master_2d"])
master_3d = pd.read_parquet(PATHS["master_3d"])

wells_feat = gpd.read_parquet(PATHS["wells_features"])
seis2d_feat = gpd.read_parquet(PATHS["seis2d_features"])
seis3d_feat = gpd.read_parquet(PATHS["seis3d_features"])
au_feat = gpd.read_parquet(PATHS["assessment_features"])
extent_feat = gpd.read_parquet(PATHS["extent_features"])

# -----------------------------
# SURFACE CONFIG
# -----------------------------
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]

surface_settings = {
    "NStopo": {"opacity": 0.25, "colorscale": "Greens"},
    "NSLCU": {"opacity": 0.35, "colorscale": "Blues"},
    "NSshublik": {"opacity": 0.45, "colorscale": "Cividis"},
    "NSbasement": {"opacity": 0.65, "colorscale": "Viridis"},
}

fig = go.Figure()

# -----------------------------
# MAIN STRUCTURAL SURFACES
# -----------------------------
surf = master_3d[master_3d["surface_name"].isin(main_surfaces)].copy()
z_col = "depth_m" if "depth_m" in surf.columns else "z_native"

for name in main_surfaces:
    df = surf[surf["surface_name"] == name].copy()

    grid = (
        df.pivot_table(index="y_3338", columns="x_3338", values=z_col, aggfunc="mean")
        .sort_index(ascending=False)
        .sort_index(axis=1)
    )

    X = grid.columns.to_numpy()
    Y = grid.index.to_numpy()
    Z = grid.to_numpy()
    X_mesh, Y_mesh = np.meshgrid(X, Y)

    fig.add_trace(
        go.Surface(
            x=X_mesh,
            y=Y_mesh,
            z=Z,
            name=name,
            opacity=surface_settings[name]["opacity"],
            colorscale=surface_settings[name]["colorscale"],
            showscale=False,
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Depth/Elev: %{z:.0f}<extra></extra>"
            ),
            visible=True,
        )
    )

# -----------------------------
# WELLS
# -----------------------------
wells_plot = wells_feat.copy()
if "lon" not in wells_plot.columns or "lat" not in wells_plot.columns:
    wells_plot = wells_plot.to_crs("EPSG:4326")
    wells_plot["lon"] = wells_plot.geometry.x
    wells_plot["lat"] = wells_plot.geometry.y

wells_xy = master_2d[master_2d["layer_name"] == "wells"][["x_3338", "y_3338", "lon", "lat"]].copy()
wells_xy = wells_xy.drop_duplicates(subset=["lon", "lat"])

wells_plot = wells_plot.merge(
    wells_xy,
    on=["lon", "lat"],
    how="left",
    suffixes=("", "_m2d")
)

fig.add_trace(
    go.Scatter3d(
        x=wells_plot["x_3338"],
        y=wells_plot["y_3338"],
        z=np.zeros(len(wells_plot)),
        mode="markers",
        name="Wells",
        marker=dict(size=3, color="black", opacity=0.9),
        customdata=np.stack([
            wells_plot.get("operator", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("field", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("au_name", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("dist_to_2d_km", pd.Series([np.nan] * len(wells_plot))).round(2).astype(str),
            wells_plot.get("in_3d", pd.Series([False] * len(wells_plot))).astype(str),
            wells_plot.get("is_exploratory", pd.Series([False] * len(wells_plot))).astype(str),
        ], axis=1),
        hovertemplate=(
            "<b>Well</b><br>"
            "Operator: %{customdata[0]}<br>"
            "Field: %{customdata[1]}<br>"
            "AU: %{customdata[2]}<br>"
            "Dist to 2D (km): %{customdata[3]}<br>"
            "Inside 3D: %{customdata[4]}<br>"
            "Exploratory: %{customdata[5]}<extra></extra>"
        ),
        visible=True,
    )
)

# -----------------------------
# 2D SEISMIC LINES
# -----------------------------
seis2d_xy = master_2d[master_2d["layer_name"] == "seismic_2d"].copy()
fig.add_trace(
    go.Scatter3d(
        x=seis2d_xy["x_3338"],
        y=seis2d_xy["y_3338"],
        z=np.zeros(len(seis2d_xy)),
        mode="lines",
        name="2D Seismic Geometry",
        line=dict(color="red", width=2),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# 2D seismic centroids for hover
if "centroid_lon" not in seis2d_feat.columns or "centroid_lat" not in seis2d_feat.columns:
    centroids = seis2d_feat.to_crs("EPSG:3338").geometry.centroid.to_crs("EPSG:4326")
    seis2d_feat["centroid_lon"] = centroids.x
    seis2d_feat["centroid_lat"] = centroids.y

# approximate x/y from centroid lon/lat by matching nearest master vertices is overkill here;
# use centroid lon/lat only for hover in a separate lightweight trace if desired

# -----------------------------
# 3D SEISMIC INVENTORY
# -----------------------------
seis3d_xy = master_2d[master_2d["layer_name"] == "seismic_3d_inventory"].copy()
fig.add_trace(
    go.Scatter3d(
        x=seis3d_xy["x_3338"],
        y=seis3d_xy["y_3338"],
        z=np.zeros(len(seis3d_xy)),
        mode="markers",
        name="3D Seismic Inventory Geometry",
        marker=dict(size=1, color="red", opacity=0.08),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# -----------------------------
# ASSESSMENT UNITS
# -----------------------------
au_xy = master_2d[master_2d["layer_name"] == "assessment_units"].copy()
fig.add_trace(
    go.Scatter3d(
        x=au_xy["x_3338"],
        y=au_xy["y_3338"],
        z=np.zeros(len(au_xy)),
        mode="lines",
        name="Assessment Units Geometry",
        line=dict(color="orange", width=1),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# -----------------------------
# NORTH SLOPE EXTENT
# -----------------------------
extent_xy = master_2d[master_2d["layer_name"] == "extent"].copy()
fig.add_trace(
    go.Scatter3d(
        x=extent_xy["x_3338"],
        y=extent_xy["y_3338"],
        z=np.zeros(len(extent_xy)),
        mode="lines",
        name="North Slope Extent",
        line=dict(color="black", width=4),
        hoverinfo="skip",
        visible=True,
    )
)

# -----------------------------
# THICKNESS / DIFFERENCE SURFACES
# -----------------------------
# these come from the master_3d parquet because your master includes
# NStopo-LCU, NSLCU-Shublik, NSshublik-basement, NStopo-basement
analysis_surfaces = [
    "NStopo-LCU",
    "NSLCU-Shublik",
    "NSshublik-basement",
    "NStopo-basement",
]

analysis_settings = {
    "NStopo-LCU": {"opacity": 0.45, "colorscale": "Magma"},
    "NSLCU-Shublik": {"opacity": 0.45, "colorscale": "Plasma"},
    "NSshublik-basement": {"opacity": 0.45, "colorscale": "Inferno"},
    "NStopo-basement": {"opacity": 0.35, "colorscale": "Turbo"},
}

analysis_df = master_3d[master_3d["surface_name"].isin(analysis_surfaces)].copy()

for name in analysis_surfaces:
    df = analysis_df[analysis_df["surface_name"] == name].copy()
    if df.empty:
        continue

    grid = (
        df.pivot_table(index="y_3338", columns="x_3338", values=z_col, aggfunc="mean")
        .sort_index(ascending=False)
        .sort_index(axis=1)
    )

    X = grid.columns.to_numpy()
    Y = grid.index.to_numpy()
    Z = grid.to_numpy()
    X_mesh, Y_mesh = np.meshgrid(X, Y)

    fig.add_trace(
        go.Surface(
            x=X_mesh,
            y=Y_mesh,
            z=Z,
            name=name,
            opacity=analysis_settings[name]["opacity"],
            colorscale=analysis_settings[name]["colorscale"],
            showscale=False,
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Value: %{z:.0f}<extra></extra>"
            ),
            visible="legendonly",
        )
    )

# -----------------------------
# LAYOUT
# -----------------------------
fig.update_layout(
    title="North Slope Master Analysis Scene",
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth / Elevation",
        zaxis=dict(autorange="reversed"),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.25),
        camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)),
    ),
    height=950,
    width=1500,
    legend=dict(itemsizing="constant"),
)

fig.show()

out_html = EXPORT_HTML_DIR / "north_slope_master_analysis_scene.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"Saved: {out_html.resolve()}")

In [ ]:
PATHS["seis2d_features"] = FEATURE_DIR / "clean_2d_seismic_features.parquet"

In [ ]:
seis2d_feat = gpd.read_parquet(PATHS["seis2d_features"])

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go

BASE_DIR = Path.cwd()

CORE_DIR = BASE_DIR / "03_data_final" / "core_layers"
FEATURE_DIR = BASE_DIR / "03_data_final" / "feature_layers"
MASTER_DIR = BASE_DIR / "03_data_final" / "master_layers"
EXPORT_HTML_DIR = BASE_DIR / "05_exports" / "html"

EXPORT_HTML_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "master_2d": MASTER_DIR / "north_slope_master_2d_layers.parquet",
    "master_3d": MASTER_DIR / "north_slope_master_3d_surfaces.parquet",
    "wells_features": FEATURE_DIR / "clean_well_locations_features_dedup.parquet",
    "extent_features": FEATURE_DIR / "north_slope_extent_features.parquet",
}

# --------------------------------------------------
# LOAD
# --------------------------------------------------
master_2d = pd.read_parquet(PATHS["master_2d"])
master_3d = pd.read_parquet(PATHS["master_3d"])
wells_feat = gpd.read_parquet(PATHS["wells_features"])
extent_feat = gpd.read_parquet(PATHS["extent_features"])

# --------------------------------------------------
# NORTH SLOPE CLIP BOUNDS
# --------------------------------------------------
extent_ll = extent_feat.to_crs("EPSG:4326")
extent_3338 = extent_feat.to_crs("EPSG:3338")

minx, miny, maxx, maxy = extent_3338.total_bounds

master_3d_ns = master_3d[
    (master_3d["x_3338"] >= minx) &
    (master_3d["x_3338"] <= maxx) &
    (master_3d["y_3338"] >= miny) &
    (master_3d["y_3338"] <= maxy)
].copy()

master_2d_ns = master_2d[
    (master_2d["x_3338"] >= minx) &
    (master_2d["x_3338"] <= maxx) &
    (master_2d["y_3338"] >= miny) &
    (master_2d["y_3338"] <= maxy)
].copy()

# --------------------------------------------------
# NORTH SLOPE WELLS ONLY
# --------------------------------------------------
wells_feat = wells_feat.to_crs("EPSG:4326")
extent_union_ll = extent_ll.geometry.union_all()
wells_ns = wells_feat[wells_feat.geometry.within(extent_union_ll)].copy()

wells_xy = master_2d_ns[master_2d_ns["layer_name"] == "wells"][["x_3338", "y_3338", "lon", "lat"]].copy()
wells_xy = wells_xy.drop_duplicates(subset=["lon", "lat"])

wells_plot = wells_ns.merge(
    wells_xy,
    on=["lon", "lat"],
    how="left",
    suffixes=("", "_m2d")
).dropna(subset=["x_3338", "y_3338"]).copy()

print("master_3d full:", len(master_3d), " clipped:", len(master_3d_ns))
print("master_2d full:", len(master_2d), " clipped:", len(master_2d_ns))
print("wells full:", len(wells_feat), " north slope:", len(wells_plot))

# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def build_grid(df, value_col):
    grid = (
        df.pivot_table(index="y_3338", columns="x_3338", values=value_col, aggfunc="mean")
        .sort_index(ascending=False)
        .sort_index(axis=1)
    )
    X = grid.columns.to_numpy()
    Y = grid.index.to_numpy()
    Z = grid.to_numpy()
    X_mesh, Y_mesh = np.meshgrid(X, Y)
    return X_mesh, Y_mesh, Z

def get_surface_df(name):
    return master_3d_ns[master_3d_ns["surface_name"] == name].copy()

def build_broken_line_arrays(df, z_value=0.0):
    """
    Build x/y/z arrays with None separators so Plotly does NOT connect
    separate features together.
    """
    if df.empty:
        return [], [], []

    df = df.sort_values(["feature_id", "vertex_order"]).copy()

    x_line, y_line, z_line = [], [], []

    for fid, sub in df.groupby("feature_id", sort=False):
        x_line.extend(sub["x_3338"].tolist())
        y_line.extend(sub["y_3338"].tolist())
        z_line.extend([z_value] * len(sub))

        # separator between features
        x_line.append(None)
        y_line.append(None)
        z_line.append(None)

    return x_line, y_line, z_line

# --------------------------------------------------
# PLOT
# --------------------------------------------------
fig = go.Figure()

# --------------------------------------------------
# MAIN STRUCTURAL HORIZONS
# --------------------------------------------------
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]
surface_settings = {
    "NStopo": {"opacity": 0.20, "colorscale": "Greens"},
    "NSLCU": {"opacity": 0.30, "colorscale": "Blues"},
    "NSshublik": {"opacity": 0.40, "colorscale": "Cividis"},
    "NSbasement": {"opacity": 0.60, "colorscale": "Viridis"},
}

for name in main_surfaces:
    df = get_surface_df(name)
    if df.empty:
        print("Skipping empty surface:", name)
        continue

    X, Y, Z = build_grid(df, "depth_m")

    fig.add_trace(
        go.Surface(
            x=X,
            y=Y,
            z=Z,
            name=name,
            showlegend=True,
            opacity=surface_settings[name]["opacity"],
            colorscale=surface_settings[name]["colorscale"],
            showscale=False,
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Depth: %{z:.0f}<extra></extra>"
            ),
            visible=True,
        )
    )

# --------------------------------------------------
# THICKNESS / DIFFERENCE LAYERS
# draped onto topography
# --------------------------------------------------
topo_df = get_surface_df("NStopo")
X_topo, Y_topo, Z_topo = build_grid(topo_df, "depth_m")

thickness_layers = [
    "NStopo-LCU",
    "NSLCU-Shublik",
    "NSshublik-basement",
    "NStopo-basement",
]

thickness_colors = {
    "NStopo-LCU": "Magma",
    "NSLCU-Shublik": "Plasma",
    "NSshublik-basement": "Inferno",
    "NStopo-basement": "Turbo",
}

for name in thickness_layers:
    df = get_surface_df(name)
    if df.empty:
        print("Skipping empty thickness layer:", name)
        continue

    _, _, Z_thick = build_grid(df, "depth_m")

    fig.add_trace(
        go.Surface(
            x=X_topo,
            y=Y_topo,
            z=Z_topo - 50,
            surfacecolor=Z_thick,
            name=name,
            showlegend=True,
            opacity=0.75,
            colorscale=thickness_colors[name],
            showscale=False,
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Topo depth: %{z:.0f}<br>"
                "Value: %{surfacecolor:.0f}<extra></extra>"
            ),
            visible="legendonly",
        )
    )

# --------------------------------------------------
# WELLS
# --------------------------------------------------
fig.add_trace(
    go.Scatter3d(
        x=wells_plot["x_3338"],
        y=wells_plot["y_3338"],
        z=np.zeros(len(wells_plot)),
        mode="markers",
        name="Wells",
        showlegend=True,
        marker=dict(size=3, color="black", opacity=0.9),
        customdata=np.stack([
            wells_plot.get("operator", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("field", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("au_name", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("dist_to_2d_km", pd.Series([np.nan] * len(wells_plot))).round(2).astype(str),
            wells_plot.get("in_3d", pd.Series([False] * len(wells_plot))).astype(str),
        ], axis=1),
        hovertemplate=(
            "<b>Well</b><br>"
            "Operator: %{customdata[0]}<br>"
            "Field: %{customdata[1]}<br>"
            "AU: %{customdata[2]}<br>"
            "Dist to 2D (km): %{customdata[3]}<br>"
            "Inside 3D: %{customdata[4]}<extra></extra>"
        ),
        visible=True,
    )
)

# --------------------------------------------------
# 2D SEISMIC - FIXED WITH BREAKS
# --------------------------------------------------
seis2d_xy = master_2d_ns[master_2d_ns["layer_name"] == "seismic_2d"].copy()
x_line, y_line, z_line = build_broken_line_arrays(seis2d_xy, z_value=0.0)

fig.add_trace(
    go.Scatter3d(
        x=x_line,
        y=y_line,
        z=z_line,
        mode="lines",
        name="2D Seismic",
        showlegend=True,
        line=dict(color="red", width=2),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# --------------------------------------------------
# 3D SEISMIC INVENTORY
# --------------------------------------------------
seis3d_xy = master_2d_ns[master_2d_ns["layer_name"] == "seismic_3d_inventory"].copy()

fig.add_trace(
    go.Scatter3d(
        x=seis3d_xy["x_3338"],
        y=seis3d_xy["y_3338"],
        z=np.zeros(len(seis3d_xy)),
        mode="markers",
        name="3D Seismic Inventory",
        showlegend=True,
        marker=dict(size=1, color="red", opacity=0.08),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# --------------------------------------------------
# ASSESSMENT UNITS - FIXED WITH BREAKS
# --------------------------------------------------
au_xy = master_2d_ns[master_2d_ns["layer_name"] == "assessment_units"].copy()
x_line, y_line, z_line = build_broken_line_arrays(au_xy, z_value=0.0)

fig.add_trace(
    go.Scatter3d(
        x=x_line,
        y=y_line,
        z=z_line,
        mode="lines",
        name="Assessment Units",
        showlegend=True,
        line=dict(color="orange", width=1),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# --------------------------------------------------
# EXTENT - FIXED WITH BREAKS
# --------------------------------------------------
extent_xy = master_2d_ns[master_2d_ns["layer_name"] == "extent"].copy()
x_line, y_line, z_line = build_broken_line_arrays(extent_xy, z_value=0.0)

fig.add_trace(
    go.Scatter3d(
        x=x_line,
        y=y_line,
        z=z_line,
        mode="lines",
        name="North Slope Extent",
        showlegend=True,
        line=dict(color="black", width=4),
        hoverinfo="skip",
        visible=True,
    )
)

# --------------------------------------------------
# LAYOUT
# --------------------------------------------------
fig.update_layout(
    title="North Slope Master Analysis Scene",
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth / Elevation",
        zaxis=dict(autorange="reversed"),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.35),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
    ),
    height=950,
    width=1550,
    legend=dict(itemsizing="constant"),
)

fig.show()

out_html = EXPORT_HTML_DIR / "north_slope_master_analysis_scene.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"Saved: {out_html.resolve()}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go

BASE_DIR = Path.cwd()

CORE_DIR = BASE_DIR / "03_data_final" / "core_layers"
FEATURE_DIR = BASE_DIR / "03_data_final" / "feature_layers"
MASTER_DIR = BASE_DIR / "03_data_final" / "master_layers"
EXPORT_HTML_DIR = BASE_DIR / "05_exports" / "html"

EXPORT_HTML_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "master_2d": MASTER_DIR / "north_slope_master_2d_layers.parquet",
    "master_3d": MASTER_DIR / "north_slope_master_3d_surfaces.parquet",
    "wells_features": FEATURE_DIR / "clean_well_locations_features_dedup.parquet",
    "seis2d_features": FEATURE_DIR / "clean_2d_seismic_features.parquet",
    "seis3d_features": FEATURE_DIR / "clean_3d_seismic_features.parquet",
    "assessment_features": FEATURE_DIR / "north_slope_assessment_units_features.parquet",
    "extent_features": FEATURE_DIR / "north_slope_extent_features.parquet",
}

# --------------------------------------------------
# LOAD
# --------------------------------------------------
master_2d = pd.read_parquet(PATHS["master_2d"])
master_3d = pd.read_parquet(PATHS["master_3d"])

wells_feat = gpd.read_parquet(PATHS["wells_features"])
seis2d_feat = gpd.read_parquet(PATHS["seis2d_features"])
seis3d_feat = gpd.read_parquet(PATHS["seis3d_features"])
au_feat = gpd.read_parquet(PATHS["assessment_features"])
extent_feat = gpd.read_parquet(PATHS["extent_features"])

# --------------------------------------------------
# NORTH SLOPE CLIP BOUNDS
# --------------------------------------------------
extent_ll = extent_feat.to_crs("EPSG:4326")
extent_3338 = extent_feat.to_crs("EPSG:3338")

minx, miny, maxx, maxy = extent_3338.total_bounds

master_3d_ns = master_3d[
    (master_3d["x_3338"] >= minx) &
    (master_3d["x_3338"] <= maxx) &
    (master_3d["y_3338"] >= miny) &
    (master_3d["y_3338"] <= maxy)
].copy()

master_2d_ns = master_2d[
    (master_2d["x_3338"] >= minx) &
    (master_2d["x_3338"] <= maxx) &
    (master_2d["y_3338"] >= miny) &
    (master_2d["y_3338"] <= maxy)
].copy()

# --------------------------------------------------
# NORTH SLOPE WELLS ONLY
# --------------------------------------------------
wells_feat = wells_feat.to_crs("EPSG:4326")
extent_union_ll = extent_ll.geometry.union_all()
wells_ns = wells_feat[wells_feat.geometry.within(extent_union_ll)].copy()

wells_xy = master_2d_ns[master_2d_ns["layer_name"] == "wells"][["x_3338", "y_3338", "lon", "lat"]].copy()
wells_xy = wells_xy.drop_duplicates(subset=["lon", "lat"])

wells_plot = wells_ns.merge(
    wells_xy,
    on=["lon", "lat"],
    how="left",
    suffixes=("", "_m2d")
).dropna(subset=["x_3338", "y_3338"]).copy()

print("master_3d full:", len(master_3d), " clipped:", len(master_3d_ns))
print("master_2d full:", len(master_2d), " clipped:", len(master_2d_ns))
print("wells full:", len(wells_feat), " north slope:", len(wells_plot))

# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def build_grid(df, value_col):
    grid = (
        df.pivot_table(index="y_3338", columns="x_3338", values=value_col, aggfunc="mean")
        .sort_index(ascending=False)
        .sort_index(axis=1)
    )
    X = grid.columns.to_numpy()
    Y = grid.index.to_numpy()
    Z = grid.to_numpy()
    X_mesh, Y_mesh = np.meshgrid(X, Y)
    return X_mesh, Y_mesh, Z

def get_surface_df(name):
    return master_3d_ns[master_3d_ns["surface_name"] == name].copy()

def build_broken_line_arrays(df, z_value=0.0):
    """
    Break line connections between separate features.
    """
    if df.empty:
        return [], [], []

    df = df.sort_values(["feature_id", "vertex_order"]).copy()

    x_line, y_line, z_line = [], [], []
    for fid, sub in df.groupby("feature_id", sort=False):
        x_line.extend(sub["x_3338"].tolist())
        y_line.extend(sub["y_3338"].tolist())
        z_line.extend([z_value] * len(sub))
        x_line.append(None)
        y_line.append(None)
        z_line.append(None)

    return x_line, y_line, z_line

def safe_fill_str(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype("string").fillna("NA")
    return df

def safe_num(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# --------------------------------------------------
# PLOT
# --------------------------------------------------
fig = go.Figure()

# --------------------------------------------------
# MAIN STRUCTURAL HORIZONS
# --------------------------------------------------
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]
surface_settings = {
    "NStopo": {"opacity": 0.20, "colorscale": "Greens"},
    "NSLCU": {"opacity": 0.30, "colorscale": "Blues"},
    "NSshublik": {"opacity": 0.40, "colorscale": "Cividis"},
    "NSbasement": {"opacity": 0.60, "colorscale": "Viridis"},
}

for name in main_surfaces:
    df = get_surface_df(name)
    if df.empty:
        print("Skipping empty surface:", name)
        continue

    X, Y, Z = build_grid(df, "depth_m")

    fig.add_trace(
        go.Surface(
            x=X,
            y=Y,
            z=Z,
            name=name,
            showlegend=True,
            opacity=surface_settings[name]["opacity"],
            colorscale=surface_settings[name]["colorscale"],
            showscale=False,
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Depth: %{z:.0f}<extra></extra>"
            ),
            visible=True,
        )
    )

# --------------------------------------------------
# THICKNESS / DIFFERENCE LAYERS
# --------------------------------------------------
topo_df = get_surface_df("NStopo")
X_topo, Y_topo, Z_topo = build_grid(topo_df, "depth_m")

thickness_layers = [
    "NStopo-LCU",
    "NSLCU-Shublik",
    "NSshublik-basement",
    "NStopo-basement",
]

thickness_colors = {
    "NStopo-LCU": "Magma",
    "NSLCU-Shublik": "Plasma",
    "NSshublik-basement": "Inferno",
    "NStopo-basement": "Turbo",
}

for name in thickness_layers:
    df = get_surface_df(name)
    if df.empty:
        print("Skipping empty thickness layer:", name)
        continue

    _, _, Z_thick = build_grid(df, "depth_m")

    fig.add_trace(
        go.Surface(
            x=X_topo,
            y=Y_topo,
            z=Z_topo - 50,
            surfacecolor=Z_thick,
            name=name,
            showlegend=True,
            opacity=0.75,
            colorscale=thickness_colors[name],
            showscale=False,
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Topo depth: %{z:.0f}<br>"
                "Value: %{surfacecolor:.0f}<extra></extra>"
            ),
            visible="legendonly",
        )
    )

# --------------------------------------------------
# WELLS
# --------------------------------------------------
wells_plot = safe_fill_str(wells_plot, ["operator", "field", "au_name"])
wells_plot = safe_num(wells_plot, ["dist_to_2d_km"])

fig.add_trace(
    go.Scatter3d(
        x=wells_plot["x_3338"],
        y=wells_plot["y_3338"],
        z=np.zeros(len(wells_plot)),
        mode="markers",
        name="Wells",
        showlegend=True,
        marker=dict(size=3, color="black", opacity=0.9),
        customdata=np.stack([
            wells_plot.get("operator", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("field", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("au_name", pd.Series(["NA"] * len(wells_plot))).astype(str),
            wells_plot.get("dist_to_2d_km", pd.Series([np.nan] * len(wells_plot))).round(2).astype(str),
            wells_plot.get("in_3d", pd.Series([False] * len(wells_plot))).astype(str),
        ], axis=1),
        hovertemplate=(
            "<b>Well</b><br>"
            "Operator: %{customdata[0]}<br>"
            "Field: %{customdata[1]}<br>"
            "AU: %{customdata[2]}<br>"
            "Dist to 2D (km): %{customdata[3]}<br>"
            "Inside 3D: %{customdata[4]}<extra></extra>"
        ),
        visible=True,
    )
)

# --------------------------------------------------
# 2D SEISMIC GEOMETRY
# --------------------------------------------------
seis2d_xy = master_2d_ns[master_2d_ns["layer_name"] == "seismic_2d"].copy()
x_line, y_line, z_line = build_broken_line_arrays(seis2d_xy, z_value=0.0)

fig.add_trace(
    go.Scatter3d(
        x=x_line,
        y=y_line,
        z=z_line,
        mode="lines",
        name="2D Seismic",
        showlegend=True,
        line=dict(color="red", width=2),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# 2D SEISMIC HOVER POINTS
seis2d_feat = seis2d_feat.to_crs("EPSG:3338").copy()
seis2d_feat = safe_fill_str(seis2d_feat, ["survey_name", "company", "type"])
seis2d_feat = safe_num(seis2d_feat, ["year_acquired", "year_released", "line_miles", "length_km", "survey_age_years"])

rep_pts = seis2d_feat.geometry.representative_point()
seis2d_feat["x_3338"] = rep_pts.x
seis2d_feat["y_3338"] = rep_pts.y

seis2d_feat_ns = seis2d_feat[
    (seis2d_feat["x_3338"] >= minx) &
    (seis2d_feat["x_3338"] <= maxx) &
    (seis2d_feat["y_3338"] >= miny) &
    (seis2d_feat["y_3338"] <= maxy)
].copy()

fig.add_trace(
    go.Scatter3d(
        x=seis2d_feat_ns["x_3338"],
        y=seis2d_feat_ns["y_3338"],
        z=np.zeros(len(seis2d_feat_ns)),
        mode="markers",
        name="2D Seismic Hover",
        showlegend=False,
        marker=dict(size=6, color="red", opacity=0.45, symbol="circle"),
        customdata=np.stack([
            seis2d_feat_ns.get("survey_name", pd.Series(["NA"] * len(seis2d_feat_ns))).astype(str),
            seis2d_feat_ns.get("company", pd.Series(["NA"] * len(seis2d_feat_ns))).astype(str),
            seis2d_feat_ns.get("year_acquired", pd.Series([np.nan] * len(seis2d_feat_ns))).astype(str),
            seis2d_feat_ns.get("year_released", pd.Series([np.nan] * len(seis2d_feat_ns))).astype(str),
            seis2d_feat_ns.get("line_miles", pd.Series([np.nan] * len(seis2d_feat_ns))).round(2).astype(str),
            seis2d_feat_ns.get("length_km", pd.Series([np.nan] * len(seis2d_feat_ns))).round(2).astype(str),
            seis2d_feat_ns.get("survey_age_years", pd.Series([np.nan] * len(seis2d_feat_ns))).round(0).astype(str),
        ], axis=1),
        hovertemplate=(
            "<b>2D Seismic Survey</b><br>"
            "Survey: %{customdata[0]}<br>"
            "Company: %{customdata[1]}<br>"
            "Year acquired: %{customdata[2]}<br>"
            "Year released: %{customdata[3]}<br>"
            "Line miles (orig): %{customdata[4]}<br>"
            "Length km (geom): %{customdata[5]}<br>"
            "Survey age: %{customdata[6]} years<extra></extra>"
        ),
        visible="legendonly",
    )
)

# --------------------------------------------------
# 3D SEISMIC INVENTORY GEOMETRY
# --------------------------------------------------
seis3d_xy = master_2d_ns[master_2d_ns["layer_name"] == "seismic_3d_inventory"].copy()

fig.add_trace(
    go.Scatter3d(
        x=seis3d_xy["x_3338"],
        y=seis3d_xy["y_3338"],
        z=np.zeros(len(seis3d_xy)),
        mode="markers",
        name="3D Seismic Inventory",
        showlegend=True,
        marker=dict(size=1, color="red", opacity=0.08),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# 3D SEISMIC HOVER POINTS
seis3d_feat = seis3d_feat.to_crs("EPSG:3338").copy()
seis3d_feat = safe_fill_str(seis3d_feat, ["survey_name", "company", "survey_type", "type"])
seis3d_feat = safe_num(seis3d_feat, ["year_acquired", "year_released", "square_miles", "area_km2", "survey_age_years"])

rep_pts = seis3d_feat.geometry.representative_point()
seis3d_feat["x_3338"] = rep_pts.x
seis3d_feat["y_3338"] = rep_pts.y

seis3d_feat_ns = seis3d_feat[
    (seis3d_feat["x_3338"] >= minx) &
    (seis3d_feat["x_3338"] <= maxx) &
    (seis3d_feat["y_3338"] >= miny) &
    (seis3d_feat["y_3338"] <= maxy)
].copy()

fig.add_trace(
    go.Scatter3d(
        x=seis3d_feat_ns["x_3338"],
        y=seis3d_feat_ns["y_3338"],
        z=np.zeros(len(seis3d_feat_ns)),
        mode="markers",
        name="3D Seismic Hover",
        showlegend=False,
        marker=dict(size=7, color="pink", opacity=0.45, symbol="diamond"),
        customdata=np.stack([
            seis3d_feat_ns.get("survey_name", pd.Series(["NA"] * len(seis3d_feat_ns))).astype(str),
            seis3d_feat_ns.get("company", pd.Series(["NA"] * len(seis3d_feat_ns))).astype(str),
            seis3d_feat_ns.get("survey_type", pd.Series(["NA"] * len(seis3d_feat_ns))).astype(str),
            seis3d_feat_ns.get("year_acquired", pd.Series([np.nan] * len(seis3d_feat_ns))).astype(str),
            seis3d_feat_ns.get("year_released", pd.Series([np.nan] * len(seis3d_feat_ns))).astype(str),
            seis3d_feat_ns.get("square_miles", pd.Series([np.nan] * len(seis3d_feat_ns))).round(2).astype(str),
            seis3d_feat_ns.get("area_km2", pd.Series([np.nan] * len(seis3d_feat_ns))).round(2).astype(str),
            seis3d_feat_ns.get("survey_age_years", pd.Series([np.nan] * len(seis3d_feat_ns))).round(0).astype(str),
        ], axis=1),
        hovertemplate=(
            "<b>3D Seismic Survey</b><br>"
            "Survey: %{customdata[0]}<br>"
            "Company: %{customdata[1]}<br>"
            "Survey type: %{customdata[2]}<br>"
            "Year acquired: %{customdata[3]}<br>"
            "Year released: %{customdata[4]}<br>"
            "Square miles (orig): %{customdata[5]}<br>"
            "Area km² (geom): %{customdata[6]}<br>"
            "Survey age: %{customdata[7]} years<extra></extra>"
        ),
        visible="legendonly",
    )
)

# --------------------------------------------------
# ASSESSMENT UNITS GEOMETRY
# use feature parquet geometry instead of master_2d exploded vertices
# --------------------------------------------------
au_feat = au_feat.to_crs("EPSG:3338").copy()
au_feat = safe_fill_str(au_feat, ["au_name", "province", "tps_name", "type"])
au_feat = safe_num(au_feat, ["area_km2"])

au_feat_ns = au_feat[au_feat.geometry.intersects(extent_3338.geometry.union_all())].copy()

# build broken polygon-outline arrays directly from polygons
x_line, y_line, z_line = [], [], []
for _, row in au_feat_ns.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue

    if geom.geom_type == "Polygon":
        parts = [geom]
    elif geom.geom_type == "MultiPolygon":
        parts = list(geom.geoms)
    else:
        continue

    for poly in parts:
        coords = list(poly.exterior.coords)
        for x, y in coords:
            x_line.append(x)
            y_line.append(y)
            z_line.append(0.0)
        x_line.append(None)
        y_line.append(None)
        z_line.append(None)

fig.add_trace(
    go.Scatter3d(
        x=x_line,
        y=y_line,
        z=z_line,
        mode="lines",
        name="Assessment Units",
        showlegend=True,
        line=dict(color="orange", width=2),
        hoverinfo="skip",
        visible="legendonly",
    )
)

# ASSESSMENT UNIT HOVER POINTS
rep_pts = au_feat_ns.geometry.representative_point()
au_feat_ns["x_3338"] = rep_pts.x
au_feat_ns["y_3338"] = rep_pts.y

fig.add_trace(
    go.Scatter3d(
        x=au_feat_ns["x_3338"],
        y=au_feat_ns["y_3338"],
        z=np.zeros(len(au_feat_ns)),
        mode="markers",
        name="Assessment Units Hover",
        showlegend=False,
        marker=dict(size=8, color="orange", opacity=0.45, symbol="square"),
        customdata=np.stack([
            au_feat_ns.get("au_name", pd.Series(["NA"] * len(au_feat_ns))).astype(str),
            au_feat_ns.get("province", pd.Series(["NA"] * len(au_feat_ns))).astype(str),
            au_feat_ns.get("tps_name", pd.Series(["NA"] * len(au_feat_ns))).astype(str),
            au_feat_ns.get("area_km2", pd.Series([np.nan] * len(au_feat_ns))).round(1).astype(str),
        ], axis=1),
        hovertemplate=(
            "<b>Assessment Unit</b><br>"
            "AU: %{customdata[0]}<br>"
            "Province: %{customdata[1]}<br>"
            "TPS: %{customdata[2]}<br>"
            "Area km²: %{customdata[3]}<extra></extra>"
        ),
        visible="legendonly",
    )
)

# --------------------------------------------------
# EXTENT
# use extent feature geometry directly
# --------------------------------------------------
extent_geom = extent_3338.copy()

x_line, y_line, z_line = [], [], []
for _, row in extent_geom.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue

    if geom.geom_type == "Polygon":
        parts = [geom]
    elif geom.geom_type == "MultiPolygon":
        parts = list(geom.geoms)
    else:
        continue

    for poly in parts:
        coords = list(poly.exterior.coords)
        for x, y in coords:
            x_line.append(x)
            y_line.append(y)
            z_line.append(0.0)
        x_line.append(None)
        y_line.append(None)
        z_line.append(None)

fig.add_trace(
    go.Scatter3d(
        x=x_line,
        y=y_line,
        z=z_line,
        mode="lines",
        name="North Slope Extent",
        showlegend=True,
        line=dict(color="black", width=4),
        hoverinfo="skip",
        visible=True,
    )
)

# --------------------------------------------------
# LAYOUT
# --------------------------------------------------
fig.update_layout(
    title="North Slope Master Analysis Scene",
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth / Elevation",
        zaxis=dict(autorange="reversed"),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.35),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
    ),
    height=950,
    width=1550,
    legend=dict(itemsizing="constant"),
)

fig.show()

out_html = EXPORT_HTML_DIR / "north_slope_master_analysis_scene.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"Saved: {out_html.resolve()}")

In [ ]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd

def geometry_to_vertices(gdf, layer_name, layer_group, default_depth=0.0):
    rows = []

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        attrs = row.drop(labels=["geometry"]).to_dict()

        def add_coords(coords, part_id=0):
            for order_id, (x, y, *rest) in enumerate(coords):
                rows.append({
                    "layer_group": layer_group,
                    "layer_name": layer_name,
                    "feature_id": idx,
                    "part_id": part_id,
                    "vertex_order": order_id,
                    "feature_type": geom.geom_type,
                    "x_3338": x,
                    "y_3338": y,
                    "depth_m": default_depth,
                    **attrs
                })

        if geom.geom_type == "Point":
            add_coords([geom.coords[0]], part_id=0)

        elif geom.geom_type == "MultiPoint":
            for part_id, part in enumerate(geom.geoms):
                add_coords([part.coords[0]], part_id=part_id)

        elif geom.geom_type == "LineString":
            add_coords(list(geom.coords)[::LINE_VERTEX_STEP], part_id=0)

        elif geom.geom_type == "MultiLineString":
            for part_id, part in enumerate(geom.geoms):
                add_coords(list(part.coords)[::LINE_VERTEX_STEP], part_id=part_id)

        elif geom.geom_type == "Polygon":
            add_coords(list(geom.exterior.coords)[::LINE_VERTEX_STEP], part_id=0)

        elif geom.geom_type == "MultiPolygon":
            for part_id, part in enumerate(geom.geoms):
                add_coords(list(part.exterior.coords)[::LINE_VERTEX_STEP], part_id=part_id)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    geom_points = gpd.GeoSeries(
        [Point(xy) for xy in zip(out["x_3338"], out["y_3338"])],
        crs=TARGET_ANALYSIS_CRS
    ).to_crs(TARGET_WEB_CRS)

    out["lon"] = geom_points.x.values
    out["lat"] = geom_points.y.values
    return out

In [ ]:
import plotly.graph_objects as go
import pandas as pd

fig = go.Figure()

# -----------------------------
# 1. structural surfaces
# -----------------------------
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]
surface_settings = {
    "NStopo": {"color": "#4daf4a", "size": 2, "opacity": 0.30},
    "NSLCU": {"color": "#377eb8", "size": 2, "opacity": 0.28},
    "NSshublik": {"color": "#ff7f00", "size": 2, "opacity": 0.28},
    "NSbasement": {"color": "#984ea3", "size": 2, "opacity": 0.22},
}

for name in main_surfaces:
    df = master_3d[master_3d["surface_name"] == name].copy()
    fig.add_trace(go.Scatter3d(
        x=df["x_3338"],
        y=df["y_3338"],
        z=df["depth_m"],
        mode="markers",
        name=name,
        marker=dict(
            size=surface_settings[name]["size"],
            color=surface_settings[name]["color"],
            opacity=surface_settings[name]["opacity"]
        ),
        hovertemplate=f"<b>{name}</b><br>X: %{{x}}<br>Y: %{{y}}<br>Depth: %{{z}} m<extra></extra>"
    ))

# -----------------------------
# 2. wells at surface
# -----------------------------
wells_df = master_2d[master_2d["layer_name"] == "wells"].copy()

fig.add_trace(go.Scatter3d(
    x=wells_df["x_3338"],
    y=wells_df["y_3338"],
    z=[0] * len(wells_df),
    mode="markers",
    name="Wells",
    marker=dict(size=3, color="black", opacity=0.85),
    customdata=np.stack([
        wells_df["operator"].fillna("NA").astype(str),
        wells_df["field"].fillna("NA").astype(str),
    ], axis=1),
    hovertemplate=(
        "<b>Well</b><br>"
        "Operator: %{customdata[0]}<br>"
        "Field: %{customdata[1]}<br>"
        "X: %{x}<br>Y: %{y}<br>Z: %{z}<extra></extra>"
    )
))

# -----------------------------
# 3. 2D seismic lines at surface
# -----------------------------
seis2d_df = master_2d[master_2d["layer_name"] == "seismic_2d"].copy()

for (fid, part_id), sub in seis2d_df.groupby(["feature_id", "part_id"], sort=False):
    sub = sub.sort_values("vertex_order")
    survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "2D seismic"

    fig.add_trace(go.Scatter3d(
        x=sub["x_3338"],
        y=sub["y_3338"],
        z=[0] * len(sub),
        mode="lines",
        name="2D Seismic",
        showlegend=False,
        line=dict(color="orange", width=3),
        customdata=np.array([[survey_name]] * len(sub)),
        hovertemplate="<b>%{customdata[0]}</b><br>2D seismic line<extra></extra>"
    ))

fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode="lines",
    name="2D Seismic",
    line=dict(color="orange", width=4)
))

# -----------------------------
# 4. 3D seismic inventory outlines at surface
# -----------------------------
seis3d_df = master_2d[master_2d["layer_name"] == "seismic_3d_inventory"].copy()

for (fid, part_id), sub in seis3d_df.groupby(["feature_id", "part_id"], sort=False):
    sub = sub.sort_values("vertex_order")
    survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "3D seismic"

    fig.add_trace(go.Scatter3d(
        x=sub["x_3338"],
        y=sub["y_3338"],
        z=[0] * len(sub),
        mode="lines",
        name="3D Seismic Inventory",
        showlegend=False,
        line=dict(color="red", width=2),
        customdata=np.array([[survey_name]] * len(sub)),
        hovertemplate="<b>%{customdata[0]}</b><br>3D seismic outline<extra></extra>"
    ))

fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode="lines",
    name="3D Seismic Inventory",
    line=dict(color="red", width=4)
))

# -----------------------------
# 5. assessment unit outlines at surface
# -----------------------------
assessment_df = master_2d[master_2d["layer_name"] == "assessment_units"].copy()

au_colors = {
    "Brookian Topset": "#1f77b4",
    "Brookian Foreset-Bottomset": "#ff7f0e",
    "Beaufortian Strata North": "#2ca02c",
    "Beaufortian Strata South": "#d62728",
    "Ellesmerian Strata North": "#9467bd",
    "Ellesmerian Strata South": "#8c564b",
}

for (fid, part_id), sub in assessment_df.groupby(["feature_id", "part_id"], sort=False):
    sub = sub.sort_values("vertex_order")
    au_name = sub["au_name"].iloc[0] if "au_name" in sub.columns else "Assessment Unit"
    color = au_colors.get(au_name, "gray")

    fig.add_trace(go.Scatter3d(
        x=sub["x_3338"],
        y=sub["y_3338"],
        z=[0] * len(sub),
        mode="lines",
        name=au_name,
        showlegend=False,
        line=dict(color=color, width=4),
        customdata=np.array([[au_name]] * len(sub)),
        hovertemplate="<b>%{customdata[0]}</b><br>Assessment unit<extra></extra>"
    ))

for au_name, color in au_colors.items():
    fig.add_trace(go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode="lines",
        name=au_name,
        line=dict(color=color, width=4)
    ))

# -----------------------------
# 6. layout
# -----------------------------
fig.update_layout(
    title="North Slope 3D Map: Surfaces + Wells + Seismic + Assessment Units",
    height=950,
    width=1400,
    scene=dict(
        xaxis_title="X (EPSG:3338)",
        yaxis_title="Y (EPSG:3338)",
        zaxis_title="Depth (m, positive downward)",
        zaxis=dict(autorange="reversed"),
        aspectmode="data",
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.9))
    ),
    legend=dict(itemsizing="constant")
)

fig.show()

In [ ]:
from pathlib import Path
import subprocess
import sys

BASE_DIR = Path.cwd()
export_dir = BASE_DIR / "notebook_exports_pdf"
export_dir.mkdir(exist_ok=True)

notebooks = [
    "01_base_map_reorganized.ipynb",
    "01_parquet_prep_cleaned.ipynb",
    "02_geologic_framework.ipynb",
    "Depth_grid_processing.ipynb",
    "North Slope Data Layer Map.ipynb",
    "Master.ipynb",
]

for nb in notebooks:
    nb_path = BASE_DIR / nb
    if not nb_path.exists():
        print(f"SKIP: {nb} not found")
        continue

    print(f"Exporting {nb} -> PDF")
    cmd = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "pdf",
        str(nb_path),
        "--output-dir",
        str(export_dir),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"DONE: {nb}")
    else:
        print(f"FAILED: {nb}")
        print(result.stderr)

print(f"\nFinished. PDF files are in:\n{export_dir}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 0. FIND PROJECT ROOT
# =========================================================
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / "03_data_final").exists() and (p / "raw_data").exists():
            return p
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()

MASTER_DIR = PROJECT_ROOT / "03_data_final" / "master_layers"
EXPORT_HTML_DIR = PROJECT_ROOT / "05_exports" / "html"
EXPORT_HTML_DIR.mkdir(parents=True, exist_ok=True)

MASTER_2D_FP = MASTER_DIR / "north_slope_master_2d_layers.parquet"
MASTER_3D_FP = MASTER_DIR / "north_slope_master_3d_surfaces.parquet"

master_2d = pd.read_parquet(MASTER_2D_FP)
master_3d = pd.read_parquet(MASTER_3D_FP)

print("master_2d:", master_2d.shape)
print("master_3d:", master_3d.shape)
print(master_2d["layer_name"].value_counts(dropna=False))

group_cols = ["feature_id", "part_id"] if "part_id" in master_2d.columns else ["feature_id"]

# =========================================================
# 1. FIGURE
# =========================================================
fig = go.Figure()

# =========================================================
# 2. SURFACES
# =========================================================
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]
surface_opacity = {
    "NStopo": 0.35,
    "NSLCU": 0.28,
    "NSshublik": 0.28,
    "NSbasement": 0.22,
}

for name in main_surfaces:
    df = master_3d[master_3d["surface_name"] == name].copy()
    if df.empty:
        continue

    xs = np.sort(df["x_3338"].unique())
    ys = np.sort(df["y_3338"].unique())[::-1]

    pivot = df.pivot(index="y_3338", columns="x_3338", values="depth_m")
    pivot = pivot.reindex(index=ys, columns=xs)

    fig.add_trace(
        go.Surface(
            x=xs,
            y=ys,
            z=pivot.values,
            name=name,
            showscale=False,
            opacity=surface_opacity[name],
            colorscale="Viridis",
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Depth: %{z:.0f} m<extra></extra>"
            )
        )
    )

# =========================================================
# 3. WELLS
# =========================================================
wells_df = master_2d[master_2d["layer_name"] == "wells"].copy()
if not wells_df.empty:
    fig.add_trace(
        go.Scatter3d(
            x=wells_df["x_3338"],
            y=wells_df["y_3338"],
            z=np.zeros(len(wells_df)),
            mode="markers",
            name="Wells",
            marker=dict(size=3, color="black", opacity=0.85),
            customdata=np.stack([
                wells_df.get("operator", pd.Series(["NA"] * len(wells_df))).fillna("NA").astype(str),
                wells_df.get("field", pd.Series(["NA"] * len(wells_df))).fillna("NA").astype(str),
            ], axis=1),
            hovertemplate=(
                "<b>Well</b><br>"
                "Operator: %{customdata[0]}<br>"
                "Field: %{customdata[1]}<br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Z: %{z:.0f}<extra></extra>"
            )
        )
    )

# =========================================================
# 4. 2D SEISMIC
# =========================================================
seis2d_df = master_2d[master_2d["layer_name"] == "seismic_2d"].copy()
if not seis2d_df.empty:
    first = True
    for _, sub in seis2d_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "2D seismic"

        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.zeros(len(sub)),
                mode="lines",
                name="2D Seismic",
                showlegend=first,
                line=dict(color="orange", width=3),
                customdata=np.array([[survey_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>2D seismic<extra></extra>"
            )
        )
        first = False

# =========================================================
# 5. 3D SEISMIC INVENTORY
# =========================================================
seis3d_df = master_2d[master_2d["layer_name"] == "seismic_3d_inventory"].copy()
if not seis3d_df.empty:
    first = True
    for _, sub in seis3d_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "3D seismic"

        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -5.0),
                mode="lines",
                name="3D Seismic Inventory",
                showlegend=first,
                line=dict(color="blue", width=2),
                customdata=np.array([[survey_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>3D seismic inventory<extra></extra>"
            )
        )
        first = False

# =========================================================
# 6. ASSESSMENT UNITS
# =========================================================
assessment_df = master_2d[master_2d["layer_name"] == "assessment_units"].copy()
au_colors = {
    "Brookian Topset": "#1f77b4",
    "Brookian Foreset-Bottomset": "#ff7f0e",
    "Beaufortian Strata North": "#2ca02c",
    "Beaufortian Strata South": "#d62728",
    "Ellesmerian Strata North": "#9467bd",
    "Ellesmerian Strata South": "#8c564b",
}

if not assessment_df.empty:
    shown = set()
    for _, sub in assessment_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        au_name = sub["au_name"].iloc[0] if "au_name" in sub.columns else "Assessment Unit"
        color = au_colors.get(au_name, "gray")
        showlegend = au_name not in shown
        shown.add(au_name)

        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -25.0),
                mode="lines",
                name=au_name,
                showlegend=showlegend,
                line=dict(color=color, width=4),
                customdata=np.array([[au_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>Assessment unit<extra></extra>"
            )
        )

# =========================================================
# 7. EXTENT
# =========================================================
extent_df = master_2d[master_2d["layer_name"] == "extent"].copy()
if not extent_df.empty:
    first = True
    for _, sub in extent_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -10.0),
                mode="lines",
                name="North Slope Extent",
                showlegend=first,
                line=dict(color="black", width=5),
                hoverinfo="skip"
            )
        )
        first = False

# =========================================================
# 8. LAYOUT
# =========================================================
fig.update_layout(
    title="North Slope Master Analysis Scene",
    height=950,
    width=1450,
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth / Elevation (m)",
        zaxis=dict(autorange="reversed"),
        aspectmode="data",
        camera=dict(eye=dict(x=1.55, y=1.55, z=0.9))
    ),
    legend=dict(itemsizing="constant")
)

fig.show()

out_html = EXPORT_HTML_DIR / "north_slope_master_analysis_scene.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print("Saved:", out_html.resolve())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 0. FIND PROJECT ROOT + LOAD MASTER TABLES
# =========================================================
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / "03_data_final").exists() and (p / "raw_data").exists():
            return p
    raise FileNotFoundError("Could not find project root containing 03_data_final and raw_data")

PROJECT_ROOT = find_project_root()

MASTER_DIR = PROJECT_ROOT / "03_data_final" / "master_layers"
MASTER_2D_FP = MASTER_DIR / "north_slope_master_2d_layers.parquet"
MASTER_3D_FP = MASTER_DIR / "north_slope_master_3d_surfaces.parquet"

if not MASTER_2D_FP.exists():
    raise FileNotFoundError(f"Missing file: {MASTER_2D_FP}")
if not MASTER_3D_FP.exists():
    raise FileNotFoundError(f"Missing file: {MASTER_3D_FP}")

master_2d = pd.read_parquet(MASTER_2D_FP)
master_3d = pd.read_parquet(MASTER_3D_FP)

print("Loaded:")
print(" master_2d:", master_2d.shape)
print(" master_3d:", master_3d.shape)

print("\nmaster_2d layer counts:")
print(master_2d["layer_name"].value_counts(dropna=False))

print("\nmaster_3d surface counts:")
print(master_3d["surface_name"].value_counts(dropna=False))

group_cols = ["feature_id", "part_id"] if "part_id" in master_2d.columns else ["feature_id"]

# =========================================================
# 1. OPTIONAL: DECIMATE 3D SURFACES MORE FOR LIGHTER PLOTTING
# =========================================================
# If your browser still struggles, increase SURFACE_STEP to 2, 3, or 4
SURFACE_STEP = 1

if SURFACE_STEP > 1:
    reduced_parts = []
    for sname, sdf in master_3d.groupby("surface_name", sort=False):
        xs = np.sort(sdf["x_3338"].unique())[::SURFACE_STEP]
        ys = np.sort(sdf["y_3338"].unique())[::SURFACE_STEP]
        sdf2 = sdf[sdf["x_3338"].isin(xs) & sdf["y_3338"].isin(ys)].copy()
        reduced_parts.append(sdf2)
    master_3d = pd.concat(reduced_parts, ignore_index=True)
    print("\nAfter extra surface reduction:", master_3d.shape)

# =========================================================
# 2. HELPERS
# =========================================================
def build_surface_grid(df, value_col="depth_m"):
    xs = np.sort(df["x_3338"].unique())
    ys = np.sort(df["y_3338"].unique())[::-1]  # reverse so north is visually consistent
    grid = df.pivot(index="y_3338", columns="x_3338", values=value_col)
    grid = grid.reindex(index=ys, columns=xs)
    return xs, ys, grid.values

def safe_series(df, col, default="NA"):
    if col in df.columns:
        return df[col].fillna(default).astype(str)
    return pd.Series([default] * len(df), index=df.index)

# =========================================================
# 3. FIGURE
# =========================================================
fig = go.Figure()

# =========================================================
# 4. MAIN STRUCTURAL SURFACES
# =========================================================
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]

surface_settings = {
    "NStopo": {"opacity": 0.30, "colorscale": "Greens",  "visible": True},
    "NSLCU": {"opacity": 0.22, "colorscale": "Blues",   "visible": "legendonly"},
    "NSshublik": {"opacity": 0.38, "colorscale": "Cividis", "visible": True},
    "NSbasement": {"opacity": 0.18, "colorscale": "Viridis", "visible": "legendonly"},
}

for name in main_surfaces:
    df = master_3d[master_3d["surface_name"] == name].copy()
    if df.empty:
        print(f"Skipping missing surface: {name}")
        continue

    xs, ys, zgrid = build_surface_grid(df, value_col="depth_m")

    fig.add_trace(
        go.Surface(
            x=xs,
            y=ys,
            z=zgrid,
            name=name,
            showlegend=True,
            visible=surface_settings[name]["visible"],
            showscale=False,
            opacity=surface_settings[name]["opacity"],
            colorscale=surface_settings[name]["colorscale"],
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Depth: %{z:.0f} m<extra></extra>"
            ),
        )
    )

# =========================================================
# 5. WELLS
# =========================================================
wells_df = master_2d[master_2d["layer_name"] == "wells"].copy()

if not wells_df.empty:
    fig.add_trace(
        go.Scatter3d(
            x=wells_df["x_3338"],
            y=wells_df["y_3338"],
            z=np.zeros(len(wells_df)),
            mode="markers",
            name="Wells",
            showlegend=True,
            visible=True,
            marker=dict(size=3, color="black", opacity=0.85),
            customdata=np.stack([
                safe_series(wells_df, "operator"),
                safe_series(wells_df, "field"),
            ], axis=1),
            hovertemplate=(
                "<b>Well</b><br>"
                "Operator: %{customdata[0]}<br>"
                "Field: %{customdata[1]}<br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Z: %{z:.0f}<extra></extra>"
            )
        )
    )

# =========================================================
# 6. 2D SEISMIC
# =========================================================
seis2d_df = master_2d[master_2d["layer_name"] == "seismic_2d"].copy()

if not seis2d_df.empty:
    first = True
    for _, sub in seis2d_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "2D seismic"
        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.zeros(len(sub)),
                mode="lines",
                name="2D Seismic",
                showlegend=first,
                visible="legendonly",
                line=dict(color="orange", width=3),
                customdata=np.array([[survey_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>2D seismic<extra></extra>"
            )
        )
        first = False

# =========================================================
# 7. 3D SEISMIC INVENTORY
# =========================================================
seis3d_df = master_2d[master_2d["layer_name"] == "seismic_3d_inventory"].copy()

if not seis3d_df.empty:
    first = True
    for _, sub in seis3d_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "3D seismic"
        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -5.0),
                mode="lines",
                name="3D Seismic Inventory",
                showlegend=first,
                visible="legendonly",
                line=dict(color="blue", width=2),
                customdata=np.array([[survey_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>3D seismic inventory<extra></extra>"
            )
        )
        first = False

# =========================================================
# 8. ASSESSMENT UNITS
# =========================================================
assessment_df = master_2d[master_2d["layer_name"] == "assessment_units"].copy()

au_colors = {
    "Brookian Topset": "#1f77b4",
    "Brookian Foreset-Bottomset": "#ff7f0e",
    "Beaufortian Strata North": "#2ca02c",
    "Beaufortian Strata South": "#d62728",
    "Ellesmerian Strata North": "#9467bd",
    "Ellesmerian Strata South": "#8c564b",
}

if not assessment_df.empty:
    shown = set()
    for _, sub in assessment_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        au_name = sub["au_name"].iloc[0] if "au_name" in sub.columns else "Assessment Unit"
        color = au_colors.get(au_name, "gray")
        showlegend = au_name not in shown
        shown.add(au_name)

        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -25.0),
                mode="lines",
                name=au_name,
                showlegend=showlegend,
                visible="legendonly",
                line=dict(color=color, width=4),
                customdata=np.array([[au_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>Assessment unit<extra></extra>"
            )
        )

# =========================================================
# 9. NORTH SLOPE EXTENT
# =========================================================
extent_df = master_2d[master_2d["layer_name"] == "extent"].copy()

if not extent_df.empty:
    first = True
    for _, sub in extent_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -10.0),
                mode="lines",
                name="North Slope Extent",
                showlegend=first,
                visible=True,
                line=dict(color="black", width=5),
                hoverinfo="skip"
            )
        )
        first = False

# =========================================================
# 10. LAYOUT
# =========================================================
fig.update_layout(
    title="North Slope Master Analysis Scene",
    height=950,
    width=1450,
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth / Elevation (m)",
        zaxis=dict(autorange="reversed"),
        aspectmode="data",
        camera=dict(eye=dict(x=1.55, y=1.55, z=0.9)),
    ),
    legend=dict(itemsizing="constant")
)

fig.show()
from pathlib import Path

# --------------------------------------------------
# 11. SAVE / REPLACE HTML EXPORT
# --------------------------------------------------
EXPORT_HTML_DIR = PROJECT_ROOT / "05_exports" / "html"
EXPORT_HTML_DIR.mkdir(parents=True, exist_ok=True)

out_html = EXPORT_HTML_DIR / "north_slope_master_analysis_scene.html"

fig.write_html(
    out_html,
    include_plotlyjs="cdn",
    full_html=True
)

print("Saved and replaced:")
print(out_html.resolve())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
from shapely.geometry import Point
from shapely.geometry.base import BaseGeometry

# =========================================================
# 0. FIND PROJECT ROOT
# =========================================================
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / "03_data_final").exists() and (p / "raw_data").exists():
            return p
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()

CORE_DIR = PROJECT_ROOT / "03_data_final" / "core_layers"
XYZ_DIR = PROJECT_ROOT / "raw_data" / "north_slope_depth_grids"
EXPORT_HTML_DIR = PROJECT_ROOT / "05_exports" / "html"
EXPORT_HTML_DIR.mkdir(parents=True, exist_ok=True)

TARGET_ANALYSIS_CRS = "EPSG:3338"
TARGET_WEB_CRS = "EPSG:4326"

PARQUET_FILES = {
    "extent": CORE_DIR / "north_slope_extent.parquet",
    "assessment_units": CORE_DIR / "north_slope_assessment_units.parquet",
    "seismic_2d": CORE_DIR / "clean_2d_seismic.parquet",
    "seismic_3d_inventory": CORE_DIR / "clean_3d_seismic.parquet",
    "wells": CORE_DIR / "clean_well_locations.parquet",
}

XYZ_SURFACES = [
    "NStopo.XYZ",
    "NSLCU.XYZ",
    "NSshublik.XYZ",
    "NSbasement.XYZ",
    "NStopo-LCU.XYZ",
    "NSLCU-Shublik.XYZ",
    "NSshublik-basement.XYZ",
    "NStopo-basement.XYZ",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
for k, v in PARQUET_FILES.items():
    print(k, "exists:", v.exists(), "->", v)
print("XYZ_DIR exists:", XYZ_DIR.exists(), "->", XYZ_DIR)

# =========================================================
# 1. LOAD + CLEAN CORE GEOMETRIES
#    no clipping
#    no line simplification
# =========================================================
def load_gdf(fp, fallback_crs=None):
    gdf = gpd.read_parquet(fp).copy()
    if "geometry" not in gdf.columns:
        raise ValueError(f"{fp.name} has no geometry column")
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry")
    if gdf.crs is None and fallback_crs is not None:
        gdf = gdf.set_crs(fallback_crs, allow_override=True)
    return gdf

def ensure_crs(gdf, target_crs):
    if gdf.crs is None:
        raise ValueError("Layer has no CRS")
    if str(gdf.crs) != target_crs:
        return gdf.to_crs(target_crs)
    return gdf.copy()

def clean_layer_geometry(gdf, layer_name):
    gdf = gdf.copy()
    gdf = gdf[gdf["geometry"].apply(lambda x: isinstance(x, BaseGeometry))].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geometry.is_valid].copy()
    print(f"{layer_name}: cleaned to {len(gdf)} rows")
    return gdf

layers = {}
for name, fp in PARQUET_FILES.items():
    if fp.exists():
        gdf = load_gdf(fp)
        gdf = ensure_crs(gdf, TARGET_ANALYSIS_CRS)
        gdf = clean_layer_geometry(gdf, name)
        layers[name] = gdf
    else:
        print("Missing:", fp)

# =========================================================
# 2. CONVERT CORE GEOMETRIES TO UNSIMPLIFIED VERTEX TABLE
#    keeps every vertex and preserves part_id
# =========================================================
LINE_VERTEX_STEP = 1  # explicit: no simplification

def geometry_to_vertices(gdf, layer_name, layer_group, default_depth=0.0):
    rows = []

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        attrs = row.drop(labels=["geometry"]).to_dict()

        def add_coords(coords, part_id=0):
            for order_id, (x, y, *rest) in enumerate(coords[::LINE_VERTEX_STEP]):
                rows.append({
                    "layer_group": layer_group,
                    "layer_name": layer_name,
                    "feature_id": idx,
                    "part_id": part_id,
                    "vertex_order": order_id,
                    "feature_type": geom.geom_type,
                    "x_3338": x,
                    "y_3338": y,
                    "depth_m": default_depth,
                    **attrs
                })

        if geom.geom_type == "Point":
            add_coords([geom.coords[0]], part_id=0)

        elif geom.geom_type == "MultiPoint":
            for part_id, part in enumerate(geom.geoms):
                add_coords([part.coords[0]], part_id=part_id)

        elif geom.geom_type == "LineString":
            add_coords(list(geom.coords), part_id=0)

        elif geom.geom_type == "MultiLineString":
            for part_id, part in enumerate(geom.geoms):
                add_coords(list(part.coords), part_id=part_id)

        elif geom.geom_type == "Polygon":
            add_coords(list(geom.exterior.coords), part_id=0)

        elif geom.geom_type == "MultiPolygon":
            for part_id, part in enumerate(geom.geoms):
                add_coords(list(part.exterior.coords), part_id=part_id)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    geom_points = gpd.GeoSeries(
        [Point(xy) for xy in zip(out["x_3338"], out["y_3338"])],
        crs=TARGET_ANALYSIS_CRS
    ).to_crs(TARGET_WEB_CRS)

    out["lon"] = geom_points.x.values
    out["lat"] = geom_points.y.values
    return out

master_2d_parts = []
layer_specs = [
    ("seismic_2d", "seismic", 0.0),
    ("seismic_3d_inventory", "seismic_inventory", 0.0),
    ("wells", "wells", 0.0),
    ("assessment_units", "geology_framework", 0.0),
    ("extent", "extent", 0.0),
]

for layer_name, layer_group, z0 in layer_specs:
    if layer_name in layers:
        part = geometry_to_vertices(
            layers[layer_name],
            layer_name=layer_name,
            layer_group=layer_group,
            default_depth=z0
        )
        part["source_file"] = PARQUET_FILES[layer_name].name
        master_2d_parts.append(part)
        print(layer_name, part.shape)

master_2d = pd.concat(master_2d_parts, ignore_index=True)
print("master_2d:", master_2d.shape)
print(master_2d["layer_name"].value_counts(dropna=False))

# =========================================================
# 3. LOAD FULL XYZ SURFACES (NO DECIMATION)
# =========================================================
def load_xyz_grid(file_path):
    rows = []
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    for line in lines[8:]:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        try:
            x = float(parts[0])
            y = float(parts[1])
            z = float(parts[2])
            rows.append((x, y, z))
        except ValueError:
            continue

    df = pd.DataFrame(rows, columns=["x_3338", "y_3338", "z_native"])
    grid = df.pivot(index="y_3338", columns="x_3338", values="z_native")
    grid = grid.sort_index(ascending=False)
    return grid

def grid_to_long_df(grid, surface_name):
    df_long = grid.stack().reset_index()
    df_long.columns = ["y_3338", "x_3338", "z_native"]
    df_long = df_long[["x_3338", "y_3338", "z_native"]].copy()
    df_long["surface_name"] = surface_name
    df_long["source_file"] = f"{surface_name}.XYZ"
    return df_long

grids = {}
for fname in XYZ_SURFACES:
    fp = XYZ_DIR / fname
    if fp.exists():
        surface_name = fp.stem
        grids[surface_name] = load_xyz_grid(fp)
        print(f"Loaded {surface_name}: {grids[surface_name].shape}")
    else:
        print(f"Missing: {fp}")

surface_parts = []
for surface_name, grid in grids.items():
    part = grid_to_long_df(grid, surface_name)
    surface_parts.append(part)

master_3d = pd.concat(surface_parts, ignore_index=True)

geom = gpd.points_from_xy(master_3d["x_3338"], master_3d["y_3338"])
gdf = gpd.GeoDataFrame(master_3d.copy(), geometry=geom, crs=TARGET_ANALYSIS_CRS)
gdf_ll = gdf.to_crs(TARGET_WEB_CRS)

master_3d["lon"] = gdf_ll.geometry.x
master_3d["lat"] = gdf_ll.geometry.y

# positive-down depth convention
master_3d["depth_m"] = -master_3d["z_native"]

master_3d = master_3d[
    ["x_3338", "y_3338", "z_native", "lon", "lat", "surface_name", "source_file", "depth_m"]
].copy()

print("master_3d:", master_3d.shape)
print(master_3d["surface_name"].value_counts(dropna=False))

# =========================================================
# 4. HELPERS FOR PLOTTING
# =========================================================
def build_surface_grid(df, value_col="depth_m"):
    xs = np.sort(df["x_3338"].unique())
    ys = np.sort(df["y_3338"].unique())[::-1]
    grid = df.pivot(index="y_3338", columns="x_3338", values=value_col)
    grid = grid.reindex(index=ys, columns=xs)
    return xs, ys, grid.values

def safe_series(df, col, default="NA"):
    if col in df.columns:
        return df[col].fillna(default).astype(str)
    return pd.Series([default] * len(df), index=df.index)

group_cols = ["feature_id", "part_id"] if "part_id" in master_2d.columns else ["feature_id"]

# =========================================================
# 5. PLOT EVERYTHING - NO CLIPPING, NO SURFACE DECIMATION
# =========================================================
fig = go.Figure()

# ---- main structural surfaces
main_surfaces = ["NStopo", "NSLCU", "NSshublik", "NSbasement"]
surface_settings = {
    "NStopo": {"opacity": 0.30, "colorscale": "Greens", "visible": True},
    "NSLCU": {"opacity": 0.22, "colorscale": "Blues", "visible": True},
    "NSshublik": {"opacity": 0.38, "colorscale": "Cividis", "visible": True},
    "NSbasement": {"opacity": 0.18, "colorscale": "Viridis", "visible": True},
}

for name in main_surfaces:
    df = master_3d[master_3d["surface_name"] == name].copy()
    if df.empty:
        continue

    xs, ys, zgrid = build_surface_grid(df, value_col="depth_m")

    fig.add_trace(
        go.Surface(
            x=xs,
            y=ys,
            z=zgrid,
            name=name,
            showlegend=True,
            visible=surface_settings[name]["visible"],
            showscale=False,
            opacity=surface_settings[name]["opacity"],
            colorscale=surface_settings[name]["colorscale"],
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Depth: %{z:.0f} m<extra></extra>"
            ),
        )
    )

# ---- thickness / difference layers too
analysis_surfaces = ["NStopo-LCU", "NSLCU-Shublik", "NSshublik-basement", "NStopo-basement"]
analysis_settings = {
    "NStopo-LCU": {"opacity": 0.35, "colorscale": "Magma", "visible": "legendonly"},
    "NSLCU-Shublik": {"opacity": 0.35, "colorscale": "Plasma", "visible": "legendonly"},
    "NSshublik-basement": {"opacity": 0.35, "colorscale": "Inferno", "visible": "legendonly"},
    "NStopo-basement": {"opacity": 0.25, "colorscale": "Turbo", "visible": "legendonly"},
}

for name in analysis_surfaces:
    df = master_3d[master_3d["surface_name"] == name].copy()
    if df.empty:
        continue

    xs, ys, zgrid = build_surface_grid(df, value_col="depth_m")

    fig.add_trace(
        go.Surface(
            x=xs,
            y=ys,
            z=zgrid,
            name=name,
            showlegend=True,
            visible=analysis_settings[name]["visible"],
            showscale=False,
            opacity=analysis_settings[name]["opacity"],
            colorscale=analysis_settings[name]["colorscale"],
            hovertemplate=(
                f"<b>{name}</b><br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Value: %{z:.0f} m<extra></extra>"
            ),
        )
    )

# ---- wells
wells_df = master_2d[master_2d["layer_name"] == "wells"].copy()
if not wells_df.empty:
    fig.add_trace(
        go.Scatter3d(
            x=wells_df["x_3338"],
            y=wells_df["y_3338"],
            z=np.zeros(len(wells_df)),
            mode="markers",
            name="Wells",
            showlegend=True,
            visible=True,
            marker=dict(size=3, color="black", opacity=0.85),
            customdata=np.stack([
                safe_series(wells_df, "operator"),
                safe_series(wells_df, "field"),
            ], axis=1),
            hovertemplate=(
                "<b>Well</b><br>"
                "Operator: %{customdata[0]}<br>"
                "Field: %{customdata[1]}<br>"
                "X: %{x:.0f}<br>"
                "Y: %{y:.0f}<br>"
                "Z: %{z:.0f}<extra></extra>"
            )
        )
    )

# ---- 2D seismic
seis2d_df = master_2d[master_2d["layer_name"] == "seismic_2d"].copy()
if not seis2d_df.empty:
    first = True
    for _, sub in seis2d_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "2D seismic"
        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.zeros(len(sub)),
                mode="lines",
                name="2D Seismic",
                showlegend=first,
                visible=True,
                line=dict(color="orange", width=3),
                customdata=np.array([[survey_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>2D seismic<extra></extra>"
            )
        )
        first = False

# ---- 3D seismic inventory
seis3d_df = master_2d[master_2d["layer_name"] == "seismic_3d_inventory"].copy()
if not seis3d_df.empty:
    first = True
    for _, sub in seis3d_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        survey_name = sub["survey_name"].iloc[0] if "survey_name" in sub.columns else "3D seismic inventory"
        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -5.0),
                mode="lines",
                name="3D Seismic Inventory",
                showlegend=first,
                visible=True,
                line=dict(color="blue", width=2),
                customdata=np.array([[survey_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>3D seismic inventory<extra></extra>"
            )
        )
        first = False

# ---- assessment units
assessment_df = master_2d[master_2d["layer_name"] == "assessment_units"].copy()
au_colors = {
    "Brookian Topset": "#1f77b4",
    "Brookian Foreset-Bottomset": "#ff7f0e",
    "Beaufortian Strata North": "#2ca02c",
    "Beaufortian Strata South": "#d62728",
    "Ellesmerian Strata North": "#9467bd",
    "Ellesmerian Strata South": "#8c564b",
}
if not assessment_df.empty:
    shown = set()
    for _, sub in assessment_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        au_name = sub["au_name"].iloc[0] if "au_name" in sub.columns else "Assessment Unit"
        color = au_colors.get(au_name, "gray")
        showlegend = au_name not in shown
        shown.add(au_name)

        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -25.0),
                mode="lines",
                name=au_name,
                showlegend=showlegend,
                visible=True,
                line=dict(color=color, width=4),
                customdata=np.array([[au_name]] * len(sub)),
                hovertemplate="<b>%{customdata[0]}</b><br>Assessment unit<extra></extra>"
            )
        )

# ---- extent
extent_df = master_2d[master_2d["layer_name"] == "extent"].copy()
if not extent_df.empty:
    first = True
    for _, sub in extent_df.groupby(group_cols, sort=False):
        sub = sub.sort_values("vertex_order")
        fig.add_trace(
            go.Scatter3d(
                x=sub["x_3338"],
                y=sub["y_3338"],
                z=np.full(len(sub), -10.0),
                mode="lines",
                name="North Slope Extent",
                showlegend=first,
                visible=True,
                line=dict(color="black", width=5),
                hoverinfo="skip"
            )
        )
        first = False

# =========================================================
# 6. LAYOUT + SAVE
# =========================================================
fig.update_layout(
    title="North Slope Master Analysis Scene - Full Geometry / No Decimation / No Clipping",
    height=1000,
    width=1550,
    scene=dict(
        xaxis_title="Projected X (EPSG:3338)",
        yaxis_title="Projected Y (EPSG:3338)",
        zaxis_title="Depth / Elevation (m)",
        zaxis=dict(autorange="reversed"),
        aspectmode="data",
        camera=dict(eye=dict(x=1.55, y=1.55, z=0.9)),
    ),
    legend=dict(itemsizing="constant")
)

fig.show()

out_html = EXPORT_HTML_DIR / "north_slope_master_analysis_scene_full_no_simplify.html"
fig.write_html(out_html, include_plotlyjs="cdn", full_html=True)

print("Saved:")
print(out_html.resolve())